In [ ]:
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Dict, Tuple, Any
import os


## Diagnoses mapper

In [ ]:
import re
import numpy as np
from typing import List, Dict, Tuple
from scipy.optimize import linear_sum_assignment

class MedicalMatcher:
    def __init__(self, json_data):
        self.pairs = json_data
    
    def match(self, diagnosis1, diagnosis2):
        diagnosis1 = diagnosis1.lower()
        diagnosis2 = diagnosis2.lower()
        
        key1 = f"{diagnosis1}|{diagnosis2}"
        key2 = f"{diagnosis2}|{diagnosis1}"
        
        if key1 in self.pairs:
            return self.pairs[key1][0]
        if key2 in self.pairs:
            return self.pairs[key2][0]
        
        words1 = set(diagnosis1.split())
        words2 = set(diagnosis2.split())
        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))
        return intersection / union if union > 0 else 0.0

def preprocess_prediction(prediction):
    """Process prediction, which is string or list"""
    if not prediction:
        return []
    
    if isinstance(prediction, list):
        if len(prediction) > 0:
            prediction_text = prediction[0]
        else:
            return []
    else:
        prediction_text = str(prediction)
    
    lines = prediction_text.split('\\n')
    diagnoses = []
    
    for line in lines:
        line = line.strip()
        match = re.match(r'^(\d+\.\s*)?([^\(\)\\n]+?)(?=\\n|\n|$|\(|\.\s|$)', line)
        if match:
            diagnosis = match.group(2).strip().lower()
            if (len(diagnosis) > 3 and 
                not diagnosis.startswith('эти') and
                not diagnosis.startswith('наиболее') and
                '...' not in diagnosis):
                diagnoses.append(diagnosis)
    
    return diagnoses[:3]

def hungarian_matching(pred_diagnoses, true_diagnoses, matcher):
    if not pred_diagnoses or not true_diagnoses:
        return 0.0
        
    similarity_matrix = np.zeros((len(pred_diagnoses), len(true_diagnoses)))
    for i, pred in enumerate(pred_diagnoses):
        for j, true in enumerate(true_diagnoses):
            similarity_matrix[i, j] = matcher.match(pred, true)
    
    row_ind, col_ind = linear_sum_assignment(-similarity_matrix)
    return np.mean(similarity_matrix[row_ind, col_ind])

def calculate_medical_score(item, matcher):
    """Calc medical score for sample"""
    if 'prediction' not in item or 'target' not in item:
        return 0.0
    
    pred_diagnoses = preprocess_prediction(item['prediction'])
    target_diagnoses = [diag.lower() for diag in item['target']]
    
    if not pred_diagnoses or not target_diagnoses:
        return 0.0
    
    return round(hungarian_matching(pred_diagnoses, target_diagnoses, matcher), 4)

def parse_log_file(log_content: str, matcher=None) -> List[Dict]:
    results = []
    current_batch = {}
    current_estimations = {}
    current_matching_scores = []
    in_estimations = False
    in_generation_metrics = False

    input_re = re.compile(r"input_texts: \['(.*?)'\]")
    target_re = re.compile(r"target_texts: \['(.*?)'\]")
    greedy_re = re.compile(r"greedy_texts: \['(.*?)'\]")
    estimator_re = re.compile(r"\('sequence', '(.*?)'\): \[(.*?)\]")
    result_re = re.compile(r"RESULT: (?:True|False) \(([\d.]+)\)")

    for line in log_content.split('\n'):
        if "================================================== NEW BATCH ==================================================" in line:
            if current_batch:
                if current_matching_scores:
                    current_estimations['matching_scores'] = current_matching_scores
                    current_estimations['max_matching_score'] = max(current_matching_scores)
                    current_estimations['min_matching_score'] = min(current_matching_scores)
                    current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
                    medical_score = calculate_medical_score(current_batch, matcher)
                    current_estimations['medical_match'] = medical_score
                
                if current_estimations:
                    current_batch['estimations'] = current_estimations
                
                results.append(current_batch)
            
            current_batch = {}
            current_estimations = {}
            current_matching_scores = []
            in_estimations = False
            in_generation_metrics = False
            continue

        if "RESULT:" in line:
            match = result_re.search(line)
            if match:
                score = float(match.group(1))
                current_matching_scores.append(score)

        if "input_texts" in line:
            match = input_re.search(line)
            if match:
                current_batch['input'] = match.group(1)

        if "target_texts" in line:
            match = target_re.search(line)
            if match:
                current_batch['target'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "greedy_texts" in line:
            match = greedy_re.search(line)
            if match:
                current_batch['prediction'] = [d.strip() for d in match.group(1).split(',') if d.strip()]

        if "Estimations:" in line:
            in_estimations = True
            in_generation_metrics = False
            continue

        if "Generation metrics:" in line:
            in_estimations = False
            in_generation_metrics = True
            continue

        if (in_estimations or in_generation_metrics) and line.strip():
            match = estimator_re.search(line)
            if match:
                estimator_name = match.group(1)
                value_str = match.group(2)
                try:
                    if ',' in value_str:
                        values = [float(v) for v in value_str.split(',')]
                        value = values[0] if values else 0.0
                    else:
                        value = float(value_str)
                    current_estimations[estimator_name] = value
                except ValueError:
                    continue

    if current_batch:
        if current_matching_scores:
            current_estimations['matching_scores'] = current_matching_scores
            current_estimations['max_matching_score'] = max(current_matching_scores)
            current_estimations['min_matching_score'] = min(current_matching_scores)
            current_estimations['minmax_matching_score'] = 1 - (max(current_matching_scores) - min(current_matching_scores))
        
            medical_score = calculate_medical_score(current_batch, matcher)
            current_estimations['medical_match'] = medical_score
        
        if current_estimations:
            current_batch['estimations'] = current_estimations
        
        results.append(current_batch)

    return results


### PRR

In [ ]:
# Universal PRR analysis for any metric
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
import os

def extract_target_metric(results: List[Dict], metric_name: str) -> List[Dict]:
    valid_results = []
    for res in results:
        if 'estimations' in res and metric_name in res['estimations']:
            res[metric_name] = res['estimations'][metric_name]
            valid_results.append(res)
    return valid_results

# --------------------------
# Normalization helpers
# --------------------------
def normalize(target: List[float]):
    min_t, max_t = np.min(target), np.max(target)
    if np.isclose(min_t, max_t):
        min_t -= 1
        max_t += 1
    target = (np.array(target) - min_t) / (max_t - min_t)
    return target

def get_random_scores(function, metrics, num_iter=200, seed=42):
    np.random.seed(seed)
    rand_scores = np.arange(len(metrics))
    value = []
    for _ in range(num_iter):
        np.random.shuffle(rand_scores)
        rand_val = function(rand_scores, metrics)
        value.append(rand_val)
    return np.mean(value)

def normalize_metric(target_score, oracle_score, random_score):
    if not (oracle_score == random_score):
        target_score = (target_score - random_score) / (oracle_score - random_score)
    return target_score

# --------------------------
# Core PRR calc
# --------------------------
def calculate_prr_area(estimator: List[float], target: List[float], max_rejection: float = 1.0) -> float:
    target = normalize(target)
    ue = np.array(estimator)
    num_obs = len(ue)
    num_rej = int(max_rejection * num_obs)

    ue_argsort = np.argsort(ue)
    sorted_metrics = np.array(target)[ue_argsort]
    cumsum = np.cumsum(sorted_metrics)[-num_rej:]
    scores = (cumsum / np.arange((num_obs - num_rej) + 1, num_obs + 1))[::-1]
    prr_score = np.sum(scores) / num_rej
    return prr_score

def compute_prr_and_normalized(estimator: List[float], target: List[float], max_rejection: float = 1.0):
    raw_score = calculate_prr_area(estimator, target, max_rejection)
    oracle_ue = 1.0 - normalize(target)
    oracle_score = calculate_prr_area(oracle_ue, target, max_rejection)
    random_score = get_random_scores(
        lambda e, t: calculate_prr_area(e, t, max_rejection), target
    )
    norm_score = normalize_metric(raw_score, oracle_score, random_score)
    return raw_score, norm_score

def compute_curve(estimator, target, max_rejection=1.0, num_points=95, do_normalize=False):
    target = normalize(target) if do_normalize else np.asarray(target)
    ue = np.asarray(estimator)
    num_obs = len(ue)

    sort_idx = np.argsort(ue)
    sorted_metrics = target[sort_idx]

    rejection_ratios = np.linspace(0, max_rejection, num_points)
    metric_means = []
    for rr in rejection_ratios:
        n_keep = int(num_obs * (1 - rr))
        metric_means.append(np.mean(sorted_metrics[:n_keep]) if n_keep > 0 else 0.0)
    return rejection_ratios, metric_means

# --------------------------
# Plotting functions
# --------------------------
def plot_prr_curves(prr_results, key, output_file, ylabel, metric_name, top_n=8):
    
    plt.figure(figsize=(12, 8))
    
    valid_estimators = [(name, vals) for name, vals in prr_results.items() 
                       if name != "Oracle" and key in vals and 'metric_means' in vals and len(vals['metric_means']) > 0]

    print(f"Plotting {key}: found {len(valid_estimators)} valid estimators")
    
    if not valid_estimators:
        print(f"No valid estimators for {key}")
        plt.close()
        return
    
    sorted_estimators = sorted(valid_estimators, key=lambda x: x[1][key], reverse=True)
    top_estimators = [name for name, _ in sorted_estimators[:top_n]]
    
    print(f"Top estimators for {key}: {top_estimators}")
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(top_estimators)))
    
    for i, name in enumerate(top_estimators):
        vals = prr_results[name]
        plt.plot(vals['rejection_ratios'], vals['metric_means'],
                 '-', linewidth=3, color=colors[i],
                 label=f"{name} ({key}={vals[key]:.3f})", alpha=0.85)
        print(f"  {name}: {len(vals['metric_means'])} points")
    
    if 'Oracle' in prr_results and 'metric_means' in prr_results['Oracle']:
        oracle_vals = prr_results['Oracle']
        plt.plot(oracle_vals['rejection_ratios'], oracle_vals['metric_means'],
                 '-', linewidth=3.5, color='black',
                 label=f"Oracle ({key}={oracle_vals[key]:.3f})")
        print(f"  Oracle: {len(oracle_vals['metric_means'])} points")
    
    plt.title(f'PRR Curves - {ylabel}', fontsize=14, fontweight='bold')
    plt.xlabel('Rejection Ratio', fontsize=12)
    plt.ylabel(f'Mean {metric_name}', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 0.9)
    plt.ylim(0.0, 1.0)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()
    print(f"Saved: {output_file}")

def plot_prr_ranking(prr_results, key, output_file, ylabel, top_n=20):
    #valid_estimators = [(name, vals) for name, vals in prr_results.items() 
    #                   if name != "Oracle" and key in vals]
    valid_estimators = [(name, vals) for name, vals in prr_results.items() 
                       if key in vals]
    
    print(f"Ranking {key}: found {len(valid_estimators)} valid estimators")
    
    if not valid_estimators:
        print(f"No valid estimators for ranking {key}")
        return
    
    sorted_estimators = sorted(valid_estimators, key=lambda x: x[1][key], reverse=True)
    top_estimators = sorted_estimators[:top_n]

    names = [name for name, _ in top_estimators]
    values = [data[key] for _, data in top_estimators]
    stds = None
    if isinstance(values[0], str):
        stds = [float(el.split("±")[1]) for el in values]
        values = [float(el.split("±")[0]) for el in values]

    plt.figure(figsize=(10, 6))
    if stds is None:
        bars = plt.barh(range(len(names)), values, color=plt.cm.Set3(np.linspace(0, 1, len(names))))
    else:
        bars = plt.barh(range(len(names)), values, xerr=stds, color=plt.cm.Set3(np.linspace(0, 1, len(names))))
    plt.yticks(range(len(names)), names, fontsize=9)
    plt.xlabel(ylabel, fontsize=12)
    plt.title(f"Top-{top_n} estimators by {ylabel}", fontsize=14, fontweight="bold")
    plt.gca().invert_yaxis()

    for i, bar in enumerate(bars):
        if stds is None:
            plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                     f"{values[i]:.3f}", va='center', fontsize=8)
        else:
            plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                     f"{values[i]:.3f}±{stds[i]:.3f}", va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {output_file}")

# --------------------------
# Main
# --------------------------


# Paper results

## PRR for k-fold data, relaxed EM, claim-level, Giga big, fixed format, fixed intro - score for full test

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
# results for test set, full_hp set data
kfold_test_pathes = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",
]

kfold_test_pathes_exp2 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033/train/2026-02-02/00-27-27/predictions/",
]

kfold_test_pathes_exp3 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2026-02-02/01-02-04/predictions/",
]

kfold_test_pathes_exp4 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1/train/2026-02-02/01-36-36/predictions/",
]

kfold_test_pathes_exp5 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_hp_prev/train/2026-02-02/02-11-00/predictions/",
]

kfold_test_pathes_exp6 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_new_hp_on_grid/train/2026-02-06/12-04-15/predictions/",
]



kfold_test_pathes_ue_nli_big_model = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
]


kfold_test_pathes_exp7 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_new_hp_on_grid_fixed/train/2026-02-09/15-44-50/predictions/",
]

kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/19-45-06/predictions/",
]

all_hp_pathes = []
all_hp_names = []
base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)

res_pathes = [kfold_test_pathes, kfold_test_pathes_exp2, kfold_test_pathes_exp3, kfold_test_pathes_exp4, kfold_test_pathes_exp5,
              kfold_test_pathes_exp6, kfold_test_pathes_ue_nli_big_model, kfold_test_pathes_exp7, kfold_test_pathes_exp8]
res_names = ["UHead_full_hp_exp1", "UHead_full_hp_exp2", "UHead_full_hp_exp3", "UHead_full_hp_exp4", "UHead_full_hp_prev_hp",
             "UHead_full_hp_new_hp_on_grid", "UHead_nli_big", "UHead_full_hp_new_hp_on_grid_fix", "UHead_full_hp_tr_head_base"]

res_pathes += all_hp_pathes
res_names += all_hp_names

scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv"

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            print(exp_buf_uhead_scores.shape)
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = []#['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


## NLI model ablation - various NLI models on test

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
# results for test set, full_hp set data
kfold_test_pathes_ue_nli_big_model = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
]

kfold_test_pathes_ue_nli_big_model_1 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/RuBioRoBERTa/predictions/",
]

kfold_test_pathes_ue_nli_big_model_2 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ruRoberta-large/predictions/",
]

kfold_test_pathes_ue_nli_big_model_3 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ClinicalBERT/predictions/",
]

kfold_test_pathes_ue_nli_big_model_hp1 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e4/ModernBERT-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp2 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e3/ModernBERT-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp3 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e4/ModernBERT-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp4 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e5/ModernBERT-large/predictions/",
]


kfold_test_pathes_ue_nli_big_model_hp1_rob = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e4/ruRoberta-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp2_rob = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e3/ruRoberta-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp3_rob = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e4/ruRoberta-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp4_rob = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e5/ruRoberta-large/predictions/",
]

res_pathes = [kfold_test_pathes_ue_nli_big_model, kfold_test_pathes_ue_nli_big_model_1,
              kfold_test_pathes_ue_nli_big_model_2, kfold_test_pathes_ue_nli_big_model_3,
              kfold_test_pathes_ue_nli_big_model_hp1, kfold_test_pathes_ue_nli_big_model_hp2,
              kfold_test_pathes_ue_nli_big_model_hp3, kfold_test_pathes_ue_nli_big_model_hp4,
              kfold_test_pathes_ue_nli_big_model_hp1_rob, kfold_test_pathes_ue_nli_big_model_hp2_rob,
              kfold_test_pathes_ue_nli_big_model_hp3_rob, kfold_test_pathes_ue_nli_big_model_hp4_rob]
res_names = ["UHead_nli_ModernBERT-large", "UHead_nli_RuBioRoBERTa",
             "UHead_nli_ruRoBERTa", "UHead_nli_ClinicalBERT",
             "UHead_nli_ModernBERT-large-hp1", "UHead_nli_ModernBERT-large-hp2",
             "UHead_nli_ModernBERT-large-hp3", "UHead_nli_ModernBERT-large-hp4",
             "UHead_nli_ruRoberta-large-hp1", "UHead_nli_ruRoberta-large-hp2",
             "UHead_nli_ruRoberta-large-hp3", "UHead_nli_ruRoberta-large-hp4",]
"""
res_pathes = [kfold_test_pathes_ue_nli_big_model, 
              kfold_test_pathes_ue_nli_big_model_hp1, kfold_test_pathes_ue_nli_big_model_hp2,
              kfold_test_pathes_ue_nli_big_model_hp3, kfold_test_pathes_ue_nli_big_model_hp4]
res_names = ["UHead_nli_ModernBERT-large", 
             "UHead_nli_ModernBERT-large-hp1", "UHead_nli_ModernBERT-large-hp2",
             "UHead_nli_ModernBERT-large-hp3", "UHead_nli_ModernBERT-large-hp4",]
"""
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv"

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/nli_ablations/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = []#['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/nli_ablations/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/nli_ablations/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


## UHead-linear, BB-approx, HP on validation

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(148),
]
all_hp_pathes = []
all_hp_names = []
base_hp_path = "../med_paper_results/gigachat_big_with_lite/hp/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
res_pathes = all_hp_pathes
res_names = all_hp_names
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv"

In [ ]:
base_file_path = "../med_data_paper/"
val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
val_data = val_data["validation"]
metric_df = val_data.to_pandas()
verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/hp/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        #metric_df = pd.read_csv(scores_path)
        metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        #metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = []#['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/hp/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite.sort_values("prr_norm_0.5")

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/hp/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


## Results for WB giga-lite, claim-level

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    print(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names
scores_path = "../med_data_paper/medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix.csv"

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=17)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=17)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
print(result_df_lite.to_latex().replace("000", ""))

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


## NLI model ablation, WB - various NLI models on test

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
# results for test set, full_hp set data
kfold_test_pathes_ue_nli_big_model = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
]

kfold_test_pathes_ue_nli_big_model_1 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/RuBioRoBERTa/predictions/",
]

kfold_test_pathes_ue_nli_big_model_2 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ruRoberta-large/predictions/",
]

kfold_test_pathes_ue_nli_big_model_3 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ClinicalBERT/predictions/",
]

kfold_test_pathes_ue_nli_big_model_hp1 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e4/ModernBERT-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp2 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e3/ModernBERT-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp3 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e4/ModernBERT-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp4 = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e5/ModernBERT-large/predictions/",
]


kfold_test_pathes_ue_nli_big_model_hp1_rob = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e4/ruRoberta-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp2_rob = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep10lr1e3/ruRoberta-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp3_rob = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e4/ruRoberta-large/predictions/",
]
kfold_test_pathes_ue_nli_big_model_hp4_rob = [
    "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/hp/ep20lr1e5/ruRoberta-large/predictions/",
]

res_pathes = [kfold_test_pathes_ue_nli_big_model, kfold_test_pathes_ue_nli_big_model_1,
              kfold_test_pathes_ue_nli_big_model_2, kfold_test_pathes_ue_nli_big_model_3,]
#              kfold_test_pathes_ue_nli_big_model_hp1, kfold_test_pathes_ue_nli_big_model_hp2,
#              kfold_test_pathes_ue_nli_big_model_hp3, kfold_test_pathes_ue_nli_big_model_hp4]
#              kfold_test_pathes_ue_nli_big_model_hp1_rob, kfold_test_pathes_ue_nli_big_model_hp2_rob,
#              kfold_test_pathes_ue_nli_big_model_hp3_rob, kfold_test_pathes_ue_nli_big_model_hp4_rob]
res_names = ["UHead_nli_ModernBERT-large", "UHead_nli_RuBioRoBERTa",
             "UHead_nli_ruRoBERTa", "UHead_nli_ClinicalBERT",]
#             "UHead_nli_ModernBERT-large-hp1", "UHead_nli_ModernBERT-large-hp2",
#             "UHead_nli_ModernBERT-large-hp3", "UHead_nli_ModernBERT-large-hp4",]
#             "UHead_nli_ruRoberta-large-hp1", "UHead_nli_ruRoberta-large-hp2",
#             "UHead_nli_ruRoberta-large-hp3", "UHead_nli_ruRoberta-large-hp4",]
"""
res_pathes = [kfold_test_pathes_ue_nli_big_model, 
              kfold_test_pathes_ue_nli_big_model_hp1, kfold_test_pathes_ue_nli_big_model_hp2,
              kfold_test_pathes_ue_nli_big_model_hp3, kfold_test_pathes_ue_nli_big_model_hp4]
res_names = ["UHead_nli_ModernBERT-large", 
             "UHead_nli_ModernBERT-large-hp1", "UHead_nli_ModernBERT-large-hp2",
             "UHead_nli_ModernBERT-large-hp3", "UHead_nli_ModernBERT-large-hp4",]
"""
scores_path = "../med_data_paper/medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix.csv"

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/nli_ablations/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = []#['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/nli_ablations/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
print(result_df_lite.to_latex().replace("000", ""))

## UHead-linear, WB, HP on validation

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(148),
]
all_hp_pathes = []
all_hp_names = []
base_hp_path = "../med_paper_results/gigachat_lite/hp/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
res_pathes = all_hp_pathes
res_names = all_hp_names
scores_path = "../med_data_paper/medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix.csv"

In [ ]:
base_file_path = "../med_data_paper/"
val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
val_data = val_data["validation"]
metric_df = val_data.to_pandas()
verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/hp/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        #metric_df = pd.read_csv(scores_path)
        metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        #metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = []#['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/hp/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite.sort_values("prr_norm_0.5")

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/hp/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


## Results for WB giga-lite, claim-level, new NLI model

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    print(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names


kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]

scores_path = "../med_data_paper/medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix_new_nli.csv"

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/new_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=17)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=17)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/new_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
# with new nli model
print(result_df_lite.to_latex().replace("000", ""))

In [ ]:
result_df_lite

## Results for WB giga-lite, claim-level, old NLI model fix + new NLI as crossencoder

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    print(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names
scores_path = "../med_data_paper/medical_data_ordered_giga_lite_fixed_format_with_claims_intro_fix_old_nli_fix.csv"

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/old_nli_fix/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=17)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=17)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_lite/old_nli_fix/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
result_df_lite

In [ ]:
# with new nli model
print(result_df_lite.to_latex().replace("000", ""))

In [ ]:
result_df_lite

## Results for BB giga-max, claim-level, new NLI model - + Giga-2-20B

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_evals"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name)
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names



base_hp_path = "../med_paper_results/giga2_20b_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name + "_giga_20b")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_giga_20b")

res_pathes += all_hp_pathes
res_names += all_hp_names
"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/giga2_20b_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_new_hp_on_grid/train/2026-02-24/14-12-50/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_giga_20b"]
"""
"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]
"""

scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue.csv"

In [ ]:
res_names

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
                               'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
                               'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
                               'LUQClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        #relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
        #                       'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
        #                       'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
        #                       'LUQClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite


| UQ method | PRR\@0.5$\uparrow$ |
|--------------|---------------|
| Eccentricity (med, Jaccard score) | 0.561 |
| UHead (approx. via GigaChat-2-lite) | 0.714  |
| UHead (approx. via GigaChat-20B-A3B) | 0.670  |


In [ ]:
# with new nli model
result_df_lite.drop(['prr_norm'], axis=1, inplace=True)
print(result_df_lite.to_latex().replace("000", ""))

### Bootstrap results to get CI

In [ ]:
uhead_path = '../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_evals/med_data_relaxed_em_target_claim/train/2026-02-12/12-53-32/predictions'
table_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue.csv"

In [ ]:
res_table = pd.read_csv(table_path)

In [ ]:
exp_buf_uhead_scores = datasets.load_from_disk(uhead_path)
exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]


In [ ]:
best_baseline_scores = res_table["estimations_EccentricityClaim_Jaccard_score"]
target_scores = res_table["filtered_claim_is_true"]
best_baseline_scores_flat_0 = [list(map(float, el.replace("[","").replace("]","").replace(",","").split())) for el in best_baseline_scores]
target_scores_flat_0 = [list(map(int, el.replace("[","").replace("]","").replace(",","").split())) for el in target_scores]

In [ ]:
best_baseline_scores_flat = []
for el, el1 in zip(best_baseline_scores_flat_0, target_scores_flat_0):
    best_baseline_scores_flat += el[:len(el1)]
target_scores_flat = []
for el in target_scores_flat_0:
    target_scores_flat += el

In [ ]:
from sklearn.utils import resample
import numpy as np
B = 5000
diffs = []
y_true = np.array(target_scores_flat)
predB = np.array(best_baseline_scores_flat)
predA = np.array(uhead_scores)

metric = lambda x, y: compute_prr_and_normalized(y, x, 0.5)[1]

for _ in range(B):

    idx = resample(
        np.arange(len(y_true)),
        replace=True,
        n_samples=len(y_true)
    )

    y = y_true[idx]

    pA = predA[idx]
    pB = predB[idx]

    mA = metric(y, pA)
    mB = metric(y, pB)

    diffs.append(mA - mB)

diffs = np.array(diffs)

ci = np.percentile(diffs, [2.5,97.5])

print(ci)

In [ ]:
pos = sum(diffs > 0)
neg = sum(diffs < 0)

In [ ]:
p_val = neg / pos * 2

In [ ]:
p_val, pos, neg

In [ ]:
from collections.abc import Callable

import numpy as np


def paired_bootstrap_test(
    y_true: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    metric: Callable[[np.ndarray, np.ndarray], float],
    n_bootstrap: int = 10_000,
    confidence_level: float = 0.95,
    random_state: int | None = None,
) -> dict[str, float | tuple[float, float] | np.ndarray]:
    """
    Compare two models evaluated on the same test examples using paired bootstrap.

    Parameters
    ----------
    y_true:
        Ground-truth labels, shape (n_samples,).
    pred_a:
        Predictions or probabilities from model A, shape (n_samples,).
    pred_b:
        Predictions or probabilities from model B, shape (n_samples,).
    metric:
        Function metric(y_true, predictions) -> scalar.
        Larger metric values are assumed to be better.
    n_bootstrap:
        Number of bootstrap resamples.
    confidence_level:
        Confidence level for the percentile interval.
    random_state:
        Seed for reproducibility.

    Returns
    -------
    Dictionary containing:
        metric_a: metric on the original test set
        metric_b: metric on the original test set
        observed_difference: metric_a - metric_b
        confidence_interval: percentile bootstrap CI
        p_value: two-sided bootstrap p-value
        bootstrap_differences: all bootstrap differences
    """
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    if not (len(y_true) == len(pred_a) == len(pred_b)):
        raise ValueError("y_true, pred_a, and pred_b must have equal lengths.")

    if len(y_true) == 0:
        raise ValueError("Input arrays must not be empty.")

    rng = np.random.default_rng(random_state)
    n_samples = len(y_true)

    metric_a = float(metric(y_true, pred_a))
    metric_b = float(metric(y_true, pred_b))
    observed_difference = metric_a - metric_b

    bootstrap_differences = []

    for _ in range(n_bootstrap):
        indices = rng.integers(
            low=0,
            high=n_samples,
            size=n_samples,
        )

        y_boot = y_true[indices]
        pred_a_boot = pred_a[indices]
        pred_b_boot = pred_b[indices]

        try:
            score_a = metric(y_boot, pred_a_boot)
            score_b = metric(y_boot, pred_b_boot)
        except ValueError:
            # Some metrics, such as AUROC, are undefined if a resample
            # contains only one class.
            continue

        if np.isfinite(score_a) and np.isfinite(score_b):
            bootstrap_differences.append(score_a - score_b)

    bootstrap_differences = np.asarray(bootstrap_differences)

    if len(bootstrap_differences) == 0:
        raise RuntimeError("The metric was undefined for every bootstrap sample.")

    alpha = 1.0 - confidence_level
    confidence_interval = tuple(
        np.quantile(
            bootstrap_differences,
            [alpha / 2, 1 - alpha / 2],
        )
    )

    # Add-one correction prevents a p-value of exactly zero.
    n_valid = len(bootstrap_differences)

    probability_nonpositive = (
        np.sum(bootstrap_differences <= 0) + 1
    ) / (n_valid + 1)

    probability_nonnegative = (
        np.sum(bootstrap_differences >= 0) + 1
    ) / (n_valid + 1)

    p_value = min(
        1.0,
        2.0 * min(probability_nonpositive, probability_nonnegative),
    )

    return {
        "metric_a": metric_a,
        "metric_b": metric_b,
        "observed_difference": observed_difference,
        "confidence_interval": confidence_interval,
        "p_value": float(p_value),
        "n_valid_bootstrap_samples": n_valid,
        "bootstrap_differences": bootstrap_differences,
    }

In [ ]:
y_true = np.array(target_scores_flat)
predB = np.array(best_baseline_scores_flat)
predA = np.array(uhead_scores)

result = paired_bootstrap_test(
    y_true=y_true,
    pred_a=predA,
    pred_b=predB,
    metric=metric,
    n_bootstrap=1_000,
    random_state=42,
)

In [ ]:
result

In [ ]:
import numpy as np
import scipy.stats

# Example metric arrays (e.g., accuracy per sample or probabilities)
#metric_a = metric(y_true, predA)
#metric_b = metric(y_true, predB)

# Calculate paired differences
#differences = metric_a - metric_b
differences = diffs
# Estimate the 95% Confidence Interval using SciPy
statistic = np.mean(differences)
ci = scipy.stats.bootstrap(
    (differences,), 
    np.mean, 
    confidence_level=0.95, 
    method='BCa' # Bias-corrected and accelerated for accuracy
)

print(f"Mean Difference: {statistic:.4f}")
print(f"95% CI: {ci.confidence_interval.low:.4f} to {ci.confidence_interval.high:.4f}")


In [ ]:
0.714 - 0.561

## Results for BB giga-max, claim-level, new NLI model - + gemma-3-4b-it

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_evals"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name)
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names



base_hp_path = "../med_paper_results/gemma_3_4b_it_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name + "_gemma_3_4b")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_gemma_3_4b")

res_pathes += all_hp_pathes
res_names += all_hp_names



base_hp_path = "../med_paper_results/gemma_3_1b_it_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name + "_gemma_3_1b")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_gemma_3_1b")

res_pathes += all_hp_pathes
res_names += all_hp_names


base_hp_path = "../med_paper_results/gemma_3_270m_it_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name + "_gemma_3_270m")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_gemma_3_270m")

res_pathes += all_hp_pathes
res_names += all_hp_names



base_hp_path = "../med_paper_results/medgemma_15_4b_it_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name + "_medgemma_15_4b")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_medgemma_15_4b")

base_hp_path = "../med_paper_results/qwen_3_4b_it_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name + "_medgemma_15_4b")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_qwen_3_4b")

res_pathes += all_hp_pathes
res_names += all_hp_names
"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/giga2_20b_approx_gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_new_hp_on_grid/train/2026-02-24/14-12-50/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_giga_20b"]
"""
"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]
"""

scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue.csv"

In [ ]:
res_names

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_w_gemma/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
                               'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
                               'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
                               'LUQClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        #relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
        #                       'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
        #                       'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
        #                       'LUQClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_w_gemma/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite


| UQ method | PRR\@0.5$\uparrow$ |
|--------------|---------------|
| Eccentricity (med, Jaccard score) (Best non-trainable method) | 0.561 |
| UHead (approx. via GigaChat-2-lite) | 0.714  |
| UHead (approx. via GigaChat-20B-A3B) | 0.670  |
| UHead (approx. via Gemma-3-4B-it) | 0.650  |
| UHead (approx. via Gemma-3-1B-it) | 0.596  |
| UHead (approx. via Gemma-3-270M-it) | 0.509  |
| UHead (approx. via MedGemma-1.5-4B-it) | 0.722  |


In [ ]:
# with new nli model
result_df_lite.drop(['prr_norm'], axis=1, inplace=True)
print(result_df_lite.to_markdown())

|                                                                    |   prr_norm_0.5 |
|:-------------------------------------------------------------------|---------------:|
| EccentricityClaim_Jaccard_score                                    |          0.561 |
| UHead (approx. via GigaChat-2-lite)                                |          0.714 |
| UHead (approx. via Gemma-3-4B-it)                                  |          0.650 |
| UHead (approx. via Qwen-3-4B-Instruct-2507)                        |          0.609 |
| UHead (approx. via Gemma-3-1B-it)                                  |          0.596 |

In [ ]:
# with new nli model
result_df_lite.drop(['prr_norm'], axis=1, inplace=True)
print(result_df_lite.to_latex().replace("000", ""))

## Results for BB giga-max, claim-level, new NLI model, test v2 with matched only

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(249),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_evals"
hp_dirs = os.listdir(base_hp_path)[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name)
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names



"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]
"""
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2.csv"
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_old_nli_v2.csv"

In [ ]:
res_names

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_v2/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
                               'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
                               'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
                               'LUQClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        #relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
        #                       'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
        #                       'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
        #                       'LUQClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_v2/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
result_df_lite

In [ ]:
# with new nli model
result_df_lite.drop(['prr_norm'], axis=1, inplace=True)
print(result_df_lite.to_latex().replace("000", ""))

## Results for BB giga-max, claim-level, new NLI model, test v2 aligned NLI

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name)
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names




"""
base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals/nli_model"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-2]
            all_hp_names.append(name + "_nli_mod")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_nli_mod")

res_pathes += all_hp_pathes
res_names += all_hp_names
"""


"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]
"""
#scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2.csv"
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2_with_aligned_nli.csv"

In [ ]:
res_names

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
len(metric_df["filtered_claim_is_true"])

In [ ]:
def main():
    log_file = '/home/jovyan/gkuzmin/llm_ue/med_data/bb_with_aligned_nli_methods/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_v2_aligned_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['NumSemSets_claim', 'NumSemSets_claim_default_nli', 'EigValLaplacian_NLI_score_entail_claim',
                               'EigValLaplacian_NLI_score_entail_claim_default_nli', 'EigValLaplacian_NLI_score_contra_claim',
                               'EigValLaplacian_NLI_score_contra_claim_default_nli', 'EigValLaplacian_Jaccard_score_claim',
                               'DegMat_NLI_score_entail_claim', 'DegMat_NLI_score_entail_claim_default_nli',
                               'DegMat_NLI_score_contra_claim', 'DegMat_NLI_score_contra_claim_default_nli',
                               'DegMat_Jaccard_score_claim', 'Eccentricity_NLI_score_entail_claim',
                               'Eccentricity_NLI_score_entail_claim_default_nli', 'Eccentricity_NLI_score_contra_claim',
                               'Eccentricity_NLI_score_contra_claim_default_nli', 'Eccentricity_Jaccard_score_claim',
                               'LexicalSimilarity_rouge1_claim', 'LexicalSimilarity_rouge2_claim',
                               'LexicalSimilarity_rougeL_claim', 'LexicalSimilarity_BLEU_claim',
                               'KernelLanguageEntropy_claim', 'KernelLanguageEntropy_claim_default_nli',
                               'SemanticEntropyEmpirical_claim', 'SemanticEntropyEmpirical_claim_default_nli',
                               'FrequencyScoringClaim', 'FrequencyScoringClaim_default_nli']
        #add_relevant_estimators = [el + '_default_nli' for el in relevant_estimators]
        #relevant_estimators += add_relevant_estimators
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        #relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
        #                       'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
        #                       'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
        #                       'LUQClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            #print(results_with_metric[0].keys())
            #print(results_with_metric[0]['estimations'].keys())
            #print(estimator)
            if "modernbert" in estimator.lower() or "relaxed_em_target_claim_new_hp_on_grid_fixed" in estimator.lower():
                print(uncertainties)
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
buf_dat = datasets.load_from_disk("../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals/nli_model/ModernBERT-large/predictions")

In [ ]:
buf_dat[1]

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_v2_aligned_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
new_test_path = '../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals/nli_model/ModernBERT-large/predictions'
nli_buf_res = datasets.load_from_disk(new_test_path)

In [ ]:
nli_buf_res[0]

In [ ]:
# results with old test set, despite the name
new_test_path = '../med_paper_results/DEBUG_gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals/nli_model/ModernBERT-large/predictions'
nli_buf_res = datasets.load_from_disk(new_test_path)

In [ ]:
nli_buf_res[0]

In [ ]:
new_test_path = '../med_paper_results/DEBUG_gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals/nli_model/ModernBERT-large/predictions'
nli_buf_res = datasets.load_from_disk(new_test_path)

In [ ]:
nli_buf_res[0]

In [ ]:
new_test_path = '../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_evals/nli_model/ModernBERT-large/predictions'
nli_buf_res = datasets.load_from_disk(new_test_path)

In [ ]:
nli_buf_res[0]

In [ ]:
result_df_lite

In [ ]:
# with new nli model
result_df_lite.drop(['prr_norm'], axis=1, inplace=True)
print(result_df_lite.to_latex().replace("000", ""))

## Results for BB giga-max, claim-level, new NLI model, test v2 aligned NLI, august fix

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals_august_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-5]
            all_hp_names.append(name)
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names




"""
base_hp_path = "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_v2_aligned_nli_evals/nli_model"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    if len(res_path) != 1:
        for buf_res_path in res_path:
            all_hp_pathes.append([buf_res_path])
            print([buf_res_path])
            
            if "nli_model" in buf_res_path:
                name = buf_res_path.split("/")[-2]
            else:
                name = buf_res_path.split("/")[-2]
            all_hp_names.append(name + "_nli_mod")
    else:
        all_hp_pathes.append(res_path)
        print(res_path)
        
        if "nli_model" in res_path:
            name = res_path[-1].split("/")[-2]
        else:
            name = res_path[-1].split("/")[-5]
        all_hp_names.append(name + "_nli_mod")

res_pathes += all_hp_pathes
res_names += all_hp_names
"""


"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]
"""
#scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2.csv"
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix_NEW_test_with_ue_med_nli_v2_with_aligned_nli_august_fix.csv"

In [ ]:
res_names

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
len(metric_df["filtered_claim_is_true"])

In [ ]:
def main():
    log_file = '/home/jovyan/gkuzmin/llm_ue/med_data/august_temp_0_res/polygraph_eval (10).log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_v2_aligned_nli_august_fix/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['NumSemSets_claim', 'NumSemSets_claim_default_nli', 'EigValLaplacian_NLI_score_entail_claim',
                               'EigValLaplacian_NLI_score_entail_claim_default_nli', 'EigValLaplacian_NLI_score_contra_claim',
                               'EigValLaplacian_NLI_score_contra_claim_default_nli', 'EigValLaplacian_Jaccard_score_claim',
                               'DegMat_NLI_score_entail_claim', 'DegMat_NLI_score_entail_claim_default_nli',
                               'DegMat_NLI_score_contra_claim', 'DegMat_NLI_score_contra_claim_default_nli',
                               'DegMat_Jaccard_score_claim', 'Eccentricity_NLI_score_entail_claim',
                               'Eccentricity_NLI_score_entail_claim_default_nli', 'Eccentricity_NLI_score_contra_claim',
                               'Eccentricity_NLI_score_contra_claim_default_nli', 'Eccentricity_Jaccard_score_claim',
                               'LexicalSimilarity_rouge1_claim', 'LexicalSimilarity_rouge2_claim',
                               'LexicalSimilarity_rougeL_claim', 'LexicalSimilarity_BLEU_claim',
                               'KernelLanguageEntropy_claim', 'KernelLanguageEntropy_claim_default_nli',
                               'SemanticEntropyEmpirical_claim', 'SemanticEntropyEmpirical_claim_default_nli',
                               'FrequencyScoringClaim', 'FrequencyScoringClaim_default_nli']
        #add_relevant_estimators = [el + '_default_nli' for el in relevant_estimators]
        #relevant_estimators += add_relevant_estimators
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        buf_res["estimations"][key] = inside_scores_array[inside_idx]
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        #relevant_estimators = ['NumSemSetsClaim', 'EigValLaplacianClaim_NLI_score_entail', 'EigValLaplacianClaim_NLI_score_contra', 'EigValLaplacianClaim_Jaccard_score',
        #                       'DegMatClaim_NLI_score_entail', 'DegMatClaim_NLI_score_contra', 'DegMatClaim_Jaccard_score', 'EccentricityClaim_NLI_score_entail',
        #                       'EccentricityClaim_NLI_score_contra', 'EccentricityClaim_Jaccard_score', 'KernelLanguageEntropyClaim', 'LexicalSimilarityClaim_rougeL',
        #                       'LUQClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            #print(results_with_metric[0].keys())
            #print(results_with_metric[0]['estimations'].keys())
            #print(estimator)
            if "modernbert" in estimator.lower() or "relaxed_em_target_claim_new_hp_on_grid_fixed" in estimator.lower():
                print(uncertainties)
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
buf_dat = datasets.load_from_disk("../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/new_test_new_nli_v2_aligned_nli_august_fix/nli_model/ModernBERT-large/predictions")

In [ ]:
buf_dat[1]

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/new_test_new_nli_v2_aligned_nli_august_fix/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

## PRR for k-fold data, relaxed EM, claim-level, Giga big, fixed format, fixed intro - holdout set best model

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(64),
]
# results for test set, full_hp set data
kfold_test_pathes_exp3 = [
    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2026-02-10/12-11-03/predictions/",
]

res_pathes = [kfold_test_pathes_exp3,]
res_names = ["UHead_full_hp_exp3"]
scores_path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv"

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/holdout_set/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            print(exp_buf_uhead_scores.shape)
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = []#['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/holdout_set/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/holdout_set/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


### Now estimate optimal threshold and manually check results

In [ ]:
from scipy.special import expit
import numpy as np
import datasets
import pandas as pd

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2026-02-10/12-11-03/predictions/"
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/12-04-28/predictions/"

TARGET_METRIC = 'filtered_claim_is_true'
exp1_uhead_scores = datasets.load_from_disk(preds_path)
exp1_uhead_scores_raw = np.array(exp1_uhead_scores['uncertainty_logits'])
exp1_uhead_scores = expit(exp1_uhead_scores_raw)#expit(exp1_uhead_scores[exp1_uhead_scores != -100])
exp1_uhead_scores_flat = expit(exp1_uhead_scores_raw[exp1_uhead_scores_raw != -100])
path = "../med_data_paper/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv"
df = pd.read_csv(path)
results = results_from_csv(df)
test_ids = list(range(len(df)))
results = [results[idx] for idx in test_ids]
# for claim level - 
metric_df = pd.read_csv("../med_data_paper/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv")
metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay, roc_curve

In [ ]:
metric_scores_flat = []
for el in metric_scores:
    metric_scores_flat += [1-inner_el for inner_el in el]

In [ ]:
precision, recall, thresholds = precision_recall_curve(metric_scores_flat, exp1_uhead_scores_flat)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = plt.figure(figsize=(4,4), layout="tight")
plt.plot(recall, precision, label=f"UHead, AP={ap_score}", zorder=1, drawstyle='steps-post')
plt.yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
plt.fill_between(recall[:30], np.zeros_like(precision[:30]), precision[:30]-.003, color='green', alpha=0.3)
plt.fill_between(recall[30:141], np.zeros_like(precision[30:141]), precision[30:141]-.003, color='yellow', alpha=0.3)
plt.fill_between(recall[141:], np.zeros_like(precision[141:]), precision[141:]-.003, color='red', alpha=0.3)
plt.text(0.2, 0.5, "Low confidence", rotation=90, va='center')
plt.text(0.65, 0.45, "Moderate confidence", rotation=90, va='center')
plt.text(0.955, 0.4, "High confidence", rotation=90, va='center')
ap_score=0.87
plt.legend(fontsize=14, loc="lower left")
plt.xlabel("Recall (positive label: 1)", fontsize=14)
plt.ylabel("Precision (positive label: 1)", fontsize=14)
fig.savefig("../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/holdout_set/fixed_format_claim_fix_intro_fix/rev_target_precision_recall_best_holdout_production.pdf")

In [ ]:
#0.37 - 141, 0.76 - 29

In [ ]:
idx = 140
recall[idx], precision[idx], thresholds[idx], 1- thresholds[idx]

In [ ]:
idx = 29
recall[idx], precision[idx], thresholds[idx], 1- thresholds[idx]

In [ ]:
thresholds

In [ ]:
PrecisionRecallDisplay.from_predictions(metric_scores_flat, exp1_uhead_scores_flat, name="UHead")

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay, roc_curve

In [ ]:
metric_scores_flat = []
for el in metric_scores:
    metric_scores_flat += el

In [ ]:
precision, recall, thresholds = precision_recall_curve(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = plt.figure(figsize=(4,4), layout="tight")
plt.plot(recall, precision, label=f"UHead, AP={ap_score}", zorder=1, drawstyle='steps-post')
ap_score=0.74
plt.legend(fontsize=14)
plt.xlabel("Recall (positive label: 1)", fontsize=14)
plt.ylabel("Precision (positive label: 1)", fontsize=14)
fig.savefig("../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/holdout_set/fixed_format_claim_fix_intro_fix/precision_recall_best_holdout.pdf")

In [ ]:
np.max(1-exp1_uhead_scores_flat), np.min(1-exp1_uhead_scores_flat)

In [ ]:
thresholds[160]

In [ ]:
#0.37 - 48, 0.76 - 160

In [ ]:
precision[48], recall[48], precision[160], recall[160]

In [ ]:
# build doc for analysis
import copy
results_table_analysis = copy.deepcopy(metric_df)

In [ ]:
results_table_analysis["uhead_scores"] = exp1_uhead_scores.tolist()

In [ ]:
exp1_uhead_scores_flat[np.argsort(exp1_uhead_scores_flat)]

In [ ]:
np.array(metric_scores_flat)[np.argsort(exp1_uhead_scores_flat)]

In [ ]:
# TODO: check if needed to use reverse values of thrs - yes, now uhead scores shows uncertainty but not confidence
def map_to_levels(row):
    scores = []
    #thr_map = {0.37: "low", 0.76: "moderate", 1.0: "high"}
    thr_map = [0.37, 0.76, 1.0]
    for el in row:
        if el[0] <= thr_map[0]:
            scores.append("high")
        elif el[0] <= thr_map[1]:
            scores.append("moderate")
        else:
            scores.append("low")
    return scores

def map_to_levels_confidence(row):
    scores = []
    #thr_map = {0.37: "low", 0.76: "moderate", 1.0: "high"}
    thr_map = [0.37, 0.76, 1.0]
    thr_map = [1-0.76, 1-0.37, 1.0]
    for el in row:
        if el[0] <= thr_map[0]:
            scores.append("high")
        elif el[0] <= thr_map[1]:
            scores.append("moderate")
        else:
            scores.append("low")
    return scores

def map_to_unc(row):
    scores = []
    for el in row:
        scores.append(np.round(el[0], 5))
    return scores

cols_to_use = ["file", "target", "input", "prediction", "claims", "filtered_claim_is_true", "uhead_scores", "confidence"]
cols_to_use = ["file", "target", "input", "prediction", "claims", "filtered_claim_is_true", "uncertainty", "confidence"]

In [ ]:
results_table_analysis["confidence"] = results_table_analysis["uhead_scores"].map(map_to_levels_confidence)
results_table_analysis["uncertainty"] = results_table_analysis["uhead_scores"].map(map_to_unc)

In [ ]:
results_table_analysis[cols_to_use]

In [ ]:
res_for_inspect = results_table_analysis[cols_to_use]
res_for_inspect.to_csv("../med_paper_results/paper_holdout_with_confidence.csv", index=False)

In [ ]:
results_table_analysis[cols_to_use]

In [ ]:
PrecisionRecallDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat, name="UHead")

In [ ]:
# plot for current prod version, before retraining
fig = plt.figure(figsize=(4,4), layout="tight")
ap_score=0.74
plt.plot(recall, precision, label=f"UHead, AP={ap_score}", zorder=1, drawstyle='steps-post')
plt.legend(fontsize=14)
plt.yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
plt.xlabel("Recall (positive label: 1)", fontsize=14)
plt.ylabel("Precision (positive label: 1)", fontsize=14)
fig.savefig("../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_big_with_lite/holdout_set/fixed_format_claim_fix_intro_fix/precision_recall_best_holdout_production.pdf")

In [ ]:
#0.37 - 48, 0.76 - 160

In [ ]:
precision[48], recall[48], precision[160], recall[160]

In [ ]:
np.max(1-exp1_uhead_scores_flat), np.min(1-exp1_uhead_scores_flat)

In [ ]:
plt.hist(1-exp1_uhead_scores_flat, bins=50)

In [ ]:
df.iloc[2], (1-exp1_uhead_scores_flat)[6:9]

In [ ]:
#exp1_uhead_scores_flat

In [ ]:
thresholds

In [ ]:
fig = PrecisionRecallDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat, name="UHead")

In [ ]:
fig.figure_

In [ ]:
PrecisionRecallDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
thresholds[-1]

In [ ]:
RocCurveDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
RocCurveDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
fpr, tpr, thr = roc_curve(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
pareto_score = np.sqrt((1-tpr)**2 + (fpr)**2)
opt_point = np.min(pareto_score)
opt_thr = thr[np.argmin(pareto_score)]
opt_point, opt_thr

In [ ]:
pareto_score = np.sqrt((1-tpr)**2)
opt_point = np.min(pareto_score)
opt_thr = thr[np.argmin(pareto_score)]
opt_point, opt_thr

In [ ]:
1-exp1_uhead_scores[0]

In [ ]:
gt_ids = np.arange(len(metric_scores_flat))
gt_ids = [gt_ids[idx] for idx, el in enumerate(metric_scores_flat) if el==1]
false_ids = np.arange(len(metric_scores_flat))
false_ids = [false_ids[idx] for idx, el in enumerate(metric_scores_flat) if el==0]
gt_ues = (1-exp1_uhead_scores_flat)[gt_ids]
false_ues = (1-exp1_uhead_scores_flat)[false_ids]

In [ ]:
np.min(gt_ues), np.max(gt_ues), np.median(gt_ues), np.percentile(gt_ues, 90), np.percentile(gt_ues, 5)

In [ ]:
np.min(false_ues), np.max(false_ues), np.median(false_ues), np.percentile(false_ues, 95), np.percentile(false_ues, 10)

In [ ]:
metric_scores_flat

In [ ]:
thr

## Results for WB giga-lite, claim-level, new NLI model + Giga-2-20B

In [ ]:
import datasets
import pandas as pd
import copy

In [ ]:
kfold_test_ids = [
    np.arange(259),
]
all_hp_pathes = []
all_hp_names = []
#    "../med_paper_results/gigachat_big_with_lite/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2026-02-01/23-52-54/predictions/",

base_hp_path = "../med_paper_results/gigachat_2_20b/enlarged_train/fixed_format_claim_fix_intro_fix"
hp_dirs = os.listdir(base_hp_path)#[:-1]
for hp_dir in hp_dirs:
    buf_dir = base_hp_path + "/" + hp_dir
    res_path = [el[0] for el in os.walk(buf_dir) if el[0].endswith("predictions")]
    all_hp_pathes.append(res_path)
    print(res_path)
    name = res_path[-1].split("/")[-5]
    all_hp_names.append(name)
print(len(all_hp_pathes))
#print(all_hp_names)
#nli_path = "../med_paper_results/gigachat_lite/enlarged_train/fixed_format_claim_fix_intro_fix/nli_model/ModernBERT-large/predictions/",
res_pathes = all_hp_pathes
res_names = all_hp_names

"""
kfold_test_pathes_exp8 = [
    "../med_paper_results/gigachat_lite/enlarged_train/tr_head/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_base/train/2026-02-11/20-04-19/predictions/",
]

res_pathes += [kfold_test_pathes_exp8]
res_names += ["UHead_tr_base"]
"""
scores_path = "../med_data_paper/medical_data_ordered_giga_2_20b_fixed_format_with_claims_intro_fix.csv"

In [ ]:
#base_file_path = "../med_data_paper/"
#val_data = datasets.load_from_disk(base_file_path + f"uhead_datasets/gigachat_big_with_lite/med_data_1_5k_full_7b_relaxed_em_target_claim_hp_split_fixed_format_intro_fix")
#val_data = val_data["validation"]
#metric_df = val_data.to_pandas()
#verified_inverse = [[int(1-x) for x in el] for el in metric_df["verified"]]
#metric_df["filtered_claim_is_true"] = verified_inverse#new_data["verified"]

In [ ]:
metric_df = pd.read_csv(scores_path)

In [ ]:
metric_df["filtered_claim_is_true"].iloc[0]

In [ ]:
metric_df

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
metric_df

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_2_20b/new_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        exp_uhead_scores_array = []
        for exp_path in res_pathes:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        path = scores_path
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv(scores_path)
        #metric_scores = [metric_df[TARGET_METRIC][idx] for idx in test_ids]
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        #print(metric_scores)
        print(len(results))
        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=14)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inside_idx, inner_el in enumerate(el):
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                for key in results[inner_idx]["estimations"].keys():
                    if key in relevant_estimators:
                        inside_scores_array = list(map(float, results[inner_idx]["estimations"][key].replace("[","").replace("]","").replace(",","").split()))
                        #print(inner_idx, inside_scores_array)
                        #print(el, inner_idx)
                        try:
                            buf_res["estimations"][key] = inside_scores_array[inside_idx]
                        except:
                            buf_res["estimations"][key] = -100
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp_uhead_scores_array[-1]), sum([len(x) for x in metric_scores]))
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results_with_metric)):
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                res_name = res_names[ue_res_idx]
                results_with_metric[idx]["estimations"][res_name] = el[idx]
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        print(results_with_metric[-1].keys(), results_with_metric[-1]["estimations"].keys())
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        

        relevant_estimators = ['MaximumClaimProbability', 'PerplexityClaim', 'MaxTokenEntropyClaim', 'SelfCertaintyClaim',
                               'PointwiseMutualInformationClaim', 'PTrueClaim', 'CCP_claim_no_context', 'CCP_claim_fact_pref',
                               'AttentionScoreClaim (layer=14)', 'FocusClaim (gamma=0.9)', 'FrequencyScoringClaim', 'TokenSARClaim']

        # for this estimators we need to flatten ue scores
        #for rel_est in relevant_estimators:
        #    print(results_with_metric["estimations"][rel_est])
        #    print(len(results_with_metric["estimations"][rel_est]))
        
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [res_names[ue_res_idx]]

        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            # for this case - remove -100
            #est_ids = np.array(uncertainties) != -100
            #uncertainties = np.array(uncertainties)[est_ids].tolist()
            #target_scores = np.array(target_scores)[est_ids].tolist()
            #print(estimator, len(uncertainties), len(target_scores), uncertainties[0], target_scores[0])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'../med_paper_results/extracted_res/claim_level_relaxed_em/gigachat_2_20b/new_nli/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
table_data = {key: {"prr_norm_0.5": np.round(float(value["prr_norm_0.5"].split("±")[0]), 3), "prr_norm": np.round(float(value["prr_norm"].split("±")[0]), 3)} for key, value in prr_05_aggregated_res.items()}

In [ ]:
result_df_lite = pd.DataFrame.from_dict(table_data).T

In [ ]:
result_df_lite = result_df_lite.drop("Oracle")

In [ ]:
result_df_lite

## PRR for k-fold data, relaxed EM, claim-level, Giga big, fixed format, holdout set

In [ ]:
import datasets
base_path_uhead_exp1 = ""
#uhead_exp1_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target/train/2025-11-14/18-30-01/predictions"
#exp1_uhead_scores = datasets.load_from_disk(uhead_exp1_path)
#exp1_uhead_scores = [el[0][0] for el in exp1_uhead_scores['uncertainty_logits']]

In [ ]:
#uhead_exp2_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target_pos_033/train/2025-11-14/19-49-37/predictions"
#exp2_uhead_scores = datasets.load_from_disk(uhead_exp2_path)
#exp2_uhead_scores = [el[0][0] for el in exp2_uhead_scores['uncertainty_logits']]

In [ ]:
#uhead_exp3_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target_attn_5_pos_5/train/2025-11-14/19-55-47/predictions"
#exp3_uhead_scores = datasets.load_from_disk(uhead_exp3_path)
#exp3_uhead_scores = [el[0][0] for el in exp3_uhead_scores['uncertainty_logits']]

In [ ]:
kfold_test_ids = [
    list(range(64)),
]
kfold_test_pathes = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim/train/2025-12-22/18-38-06/predictions/",
]

kfold_test_pathes_exp2 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim_pos_033/train/2025-12-22/18-39-25/predictions/",
]

kfold_test_pathes_exp3 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-22/18-40-38/predictions/",
]

kfold_test_pathes_exp4 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim_pos_1/train/2025-12-22/18-41-54/predictions/",
]


# now add enlarged train (with same test splits)
enlarged_train_kfold_test_pathes_exp1 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-22/18-43-06/predictions/",
]


In [ ]:
x=1

In [ ]:
import pandas as pd
import copy

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
#kfold_test_ids

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'./med_ue_res/claim_level_relaxed_em/giga_big/fixed_format/holdout_test/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        uhead_exp1_path = kfold_test_pathes[kfold_idx]
        exp1_uhead_scores = datasets.load_from_disk(uhead_exp1_path)
        exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
        exp1_uhead_scores = exp1_uhead_scores[exp1_uhead_scores != -100]
        uhead_exp2_path = kfold_test_pathes_exp2[kfold_idx]
        exp2_uhead_scores = datasets.load_from_disk(uhead_exp2_path)
        exp2_uhead_scores = np.array(exp2_uhead_scores['uncertainty_logits'])
        exp2_uhead_scores = exp2_uhead_scores[exp2_uhead_scores != -100]
        uhead_exp3_path = kfold_test_pathes_exp3[kfold_idx]
        exp3_uhead_scores = datasets.load_from_disk(uhead_exp3_path)
        exp3_uhead_scores = np.array(exp3_uhead_scores['uncertainty_logits'])
        exp3_uhead_scores = exp3_uhead_scores[exp3_uhead_scores != -100]
        uhead_exp4_path = kfold_test_pathes_exp4[kfold_idx]
        exp4_uhead_scores = datasets.load_from_disk(uhead_exp4_path)
        exp4_uhead_scores = np.array(exp4_uhead_scores['uncertainty_logits'])
        exp4_uhead_scores = exp4_uhead_scores[exp4_uhead_scores != -100]

        exp_uhead_scores_array = []
        for exp_path in [enlarged_train_kfold_test_pathes_exp1]:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))

        
        path = "./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv"
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv("./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv")
        #metric_scores = list(metric_df[TARGET_METRIC])
        #print(metric_scores)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        #metric_scores = [[int(1-x) for x in el] for el in metric_scores]
        #metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(".","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp2_uhead_scores), sum([len(x) for x in metric_scores]))
        #res_metr_only = [el[TARGET_METRIC] for el in results_with_metric]
        #print(res_metr_only)
        #results_with_metric = results
        #for inner_idx, res in enumerate(results_with_metric):
        #    res[TARGET_METRIC] = metric_scores[inner_idx]
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results)):
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_0"] = exp1_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_1"] = exp2_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_2"] = exp3_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_3"] = exp4_uhead_scores[idx]
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                results_with_metric[idx]["estimations"][f"UHead_exp_full_hp_{ue_res_idx+4}"] = el[idx]
            #for ue_res_idx, el in enumerate(exp_uhead_scores_array_full_hp):
            #    results_with_metric[idx]["estimations"][f"UHead_exp_full_hp_{ue_res_idx}"] = el[idx]

        
        '''
        metric_df = pd.read_csv("./med_data/medical_data_ordered_with_claims.csv")
        metric_scores = np.array(metric_df[TARGET_METRIC])[test_ids].flatten()


        def extract_target_metric(results: List[Dict], metric_name: str) -> List[Dict]:
                    valid_results = []
            for res in results:
                if 'estimations' in res and metric_name in res['estimations']:
                    res[metric_name] = res['estimations'][metric_name]
                    valid_results.append(res)
            return valid_results
        #print(results)
        results_with_metric = results
        for inner_idx, res in enumerate(results_with_metric):
            res[TARGET_METRIC] = metric_scores[inner_idx]
        print(results_with_metric[-1])
        '''

        #results_with_metric = extract_target_metric(results, TARGET_METRIC)
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = ['UHead_exp_full_hp_0', 'UHead_exp_full_hp_1', 'UHead_exp_full_hp_2', 'UHead_exp_full_hp_3']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            relevant_estimators += [f'UHead_exp_full_hp_{ue_res_idx+4}']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array_full_hp):
        #    relevant_estimators += [f"UHead_exp_full_hp_{ue_res_idx}"]
        # ['NumSemSets', 'EigValLaplacian_NLI_score_entail',  'EigValLaplacian_NLI_score_contra',
        #                        'EigValLaplacian_NLI_Jaccard_score','DegMat_NLI_score_entail', 'DegMat_NLI_score_contra',
        #                        'DegMat_Jaccard_score', 'Eccentricity_NLI_score_entail', 'Eccentricity_NLI_score_contra',
        #                        'Eccentricity_Jaccard_score', 'LexicalSimilarity_rouge1', 'LUQ',
        #                        'LexicalSimilarity_rouge2', 'LexicalSimilarity_rougeL', 'LexicalSimilarity_BLEU',
        #                        'KernelLanguageEntropy', 'SemanticDensity']
        
        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

### Now estimate optimal threshold and manually check results

In [ ]:
from scipy.special import expit

In [ ]:
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim/train/2025-12-22/18-38-06/predictions/"
exp1_uhead_scores = datasets.load_from_disk(preds_path)
exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
exp1_uhead_scores = expit(exp1_uhead_scores)#expit(exp1_uhead_scores[exp1_uhead_scores != -100])
exp1_uhead_scores_flat = expit(exp1_uhead_scores[exp1_uhead_scores != -100])
path = "./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv"
df = pd.read_csv(path)
results = results_from_csv(df)
test_ids = list(range(len(df)))
results = [results[idx] for idx in test_ids]
# for claim level - 
metric_df = pd.read_csv("./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv")
metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay, roc_curve

In [ ]:
metric_scores_flat = []
for el in metric_scores:
    metric_scores_flat += el

In [ ]:
precision, recall, thresholds = precision_recall_curve(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
PrecisionRecallDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
RocCurveDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
fpr, tpr, thr = roc_curve(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
pareto_score = np.sqrt((1-tpr)**2 + (fpr)**2)
opt_point = np.min(pareto_score)
opt_thr = thr[np.argmin(pareto_score)]
opt_point, opt_thr

In [ ]:
prob_idx = 4
best_thr = 0.3083
print(results[prob_idx]["input"])
print("Pred:", results[prob_idx]["prediction"])
print("Target:", results[prob_idx]["target"])
print(results[prob_idx]["estimations"])

print(1-exp1_uhead_scores[prob_idx])

In [ ]:
results[6]

In [ ]:
exp1_uhead_scores

## PRR for k-fold data, relaxed EM, claim-level, Giga big, fixed format, fixed intro

In [ ]:
import datasets
base_path_uhead_exp1 = ""
#uhead_exp1_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target/train/2025-11-14/18-30-01/predictions"
#exp1_uhead_scores = datasets.load_from_disk(uhead_exp1_path)
#exp1_uhead_scores = [el[0][0] for el in exp1_uhead_scores['uncertainty_logits']]

In [ ]:
#uhead_exp2_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target_pos_033/train/2025-11-14/19-49-37/predictions"
#exp2_uhead_scores = datasets.load_from_disk(uhead_exp2_path)
#exp2_uhead_scores = [el[0][0] for el in exp2_uhead_scores['uncertainty_logits']]

In [ ]:
#uhead_exp3_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target_attn_5_pos_5/train/2025-11-14/19-55-47/predictions"
#exp3_uhead_scores = datasets.load_from_disk(uhead_exp3_path)
#exp3_uhead_scores = [el[0][0] for el in exp3_uhead_scores['uncertainty_logits']]

In [ ]:
kfold_test_ids = [
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51],
    [52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103],
    [104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155],
    [156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207],
    [208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258],
]
kfold_test_pathes = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_cv_0/train/2025-12-24/04-43-07/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_cv_1/train/2025-12-24/05-15-24/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_cv_2/train/2025-12-24/05-47-31/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_cv_3/train/2025-12-24/06-20-34/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_cv_4/train/2025-12-24/06-54-26/predictions/",
]

kfold_test_pathes_exp2 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033_cv_0/train/2025-12-24/04-51-08/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033_cv_1/train/2025-12-24/05-23-37/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033_cv_2/train/2025-12-24/05-55-50/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033_cv_3/train/2025-12-24/06-29-16/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033_cv_4/train/2025-12-24/07-02-28/predictions/",
]

kfold_test_pathes_exp3 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_0/train/2025-12-24/04-59-12/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_1/train/2025-12-24/05-31-27/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_2/train/2025-12-24/06-03-56/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_3/train/2025-12-24/06-37-23/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_4/train/2025-12-24/07-11-02/predictions/",
]

kfold_test_pathes_exp4 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1_cv_0/train/2025-12-24/05-07-00/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1_cv_1/train/2025-12-24/05-39-15/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1_cv_2/train/2025-12-24/06-12-12/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1_cv_3/train/2025-12-24/06-45-37/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1_cv_4/train/2025-12-24/07-18-39/predictions/",
]

"""
# now add enlarged train (with same test splits)
enlarged_train_kfold_test_pathes_exp1 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_cv_0/train/2025-12-17/19-50-45/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_cv_1/train/2025-12-17/22-40-13/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_cv_2/train/2025-12-18/01-26-09/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_cv_3/train/2025-12-18/04-09-38/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_cv_4/train/2025-12-18/07-20-26/predictions/",
]

enlarged_train_kfold_test_pathes_exp2 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_033_cv_0/train/2025-12-17/20-35-09/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_033_cv_1/train/2025-12-17/23-22-12/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_033_cv_2/train/2025-12-18/02-07-01/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_033_cv_3/train/2025-12-18/04-50-34/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_033_cv_4/train/2025-12-18/08-01-18/predictions/",
]

enlarged_train_kfold_test_pathes_exp3 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_0/train/2025-12-17/21-16-41/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_1/train/2025-12-18/00-03-59/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_2/train/2025-12-18/02-48-30/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_3/train/2025-12-18/05-31-57/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_attn_5_pos_5_cv_4/train/2025-12-18/08-42-56/predictions/",
]

enlarged_train_kfold_test_pathes_exp4 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_1_cv_0/train/2025-12-17/21-58-38/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_1_cv_1/train/2025-12-18/00-45-05/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_1_cv_2/train/2025-12-18/03-28-52/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_1_cv_3/train/2025-12-18/06-26-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim_pos_1_cv_4/train/2025-12-18/09-23-45/predictions/",
]

"""



kfold_test_pathes_full_hp_exp1 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2025-12-24/01-32-55/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2025-12-24/01-32-55/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2025-12-24/01-32-55/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2025-12-24/01-32-55/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim/train/2025-12-24/01-32-55/predictions/",
]


kfold_test_pathes_full_hp_exp2 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033/train/2025-12-24/02-08-53/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033/train/2025-12-24/02-08-53/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033/train/2025-12-24/02-08-53/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033/train/2025-12-24/02-08-53/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_033/train/2025-12-24/02-08-53/predictions/",
]


kfold_test_pathes_full_hp_exp3 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-24/02-45-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-24/02-45-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-24/02-45-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-24/02-45-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-24/02-45-06/predictions/",
]


kfold_test_pathes_full_hp_exp4 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/03-21-31/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/03-21-31/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/03-21-31/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/03-21-31/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/03-21-31/predictions/",
]


kfold_test_pathes_full_hp_exp5 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-24/03-57-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-24/03-57-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-24/03-57-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-24/03-57-06/predictions/",
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-24/03-57-06/predictions/",
]


In [ ]:
x=1

In [ ]:
import pandas as pd
import copy

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'./med_ue_res/claim_level_relaxed_em/giga_big/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        uhead_exp1_path = kfold_test_pathes[kfold_idx]
        exp1_uhead_scores = datasets.load_from_disk(uhead_exp1_path)
        exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
        exp1_uhead_scores = exp1_uhead_scores[exp1_uhead_scores != -100]
        uhead_exp2_path = kfold_test_pathes_exp2[kfold_idx]
        exp2_uhead_scores = datasets.load_from_disk(uhead_exp2_path)
        exp2_uhead_scores = np.array(exp2_uhead_scores['uncertainty_logits'])
        exp2_uhead_scores = exp2_uhead_scores[exp2_uhead_scores != -100]
        uhead_exp3_path = kfold_test_pathes_exp3[kfold_idx]
        exp3_uhead_scores = datasets.load_from_disk(uhead_exp3_path)
        exp3_uhead_scores = np.array(exp3_uhead_scores['uncertainty_logits'])
        exp3_uhead_scores = exp3_uhead_scores[exp3_uhead_scores != -100]
        uhead_exp4_path = kfold_test_pathes_exp4[kfold_idx]
        exp4_uhead_scores = datasets.load_from_disk(uhead_exp4_path)
        exp4_uhead_scores = np.array(exp4_uhead_scores['uncertainty_logits'])
        exp4_uhead_scores = exp4_uhead_scores[exp4_uhead_scores != -100]

        """
        exp_uhead_scores_array = []
        for exp_path in [enlarged_train_kfold_test_pathes_exp1, enlarged_train_kfold_test_pathes_exp2,                         
                         enlarged_train_kfold_test_pathes_exp3, enlarged_train_kfold_test_pathes_exp4,]:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))
        """

        exp_uhead_scores_array_full_hp = []
        for exp_path in [kfold_test_pathes_full_hp_exp1, kfold_test_pathes_full_hp_exp2,                         
                         kfold_test_pathes_full_hp_exp3, kfold_test_pathes_full_hp_exp4, kfold_test_pathes_full_hp_exp5]:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array([exp_buf_uhead_scores['uncertainty_logits'][idx] for idx in test_ids])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array_full_hp.append(copy.deepcopy(exp_buf_uhead_scores))
        
        path = "./med_data/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv"
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv("./med_data/medical_data_ordered_giga_big_fixed_format_with_claims_intro_fix.csv")
        #metric_scores = list(metric_df[TARGET_METRIC])
        #print(metric_scores)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        #metric_scores = [[int(1-x) for x in el] for el in metric_scores]
        #metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(".","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp2_uhead_scores), sum([len(x) for x in metric_scores]))
        #res_metr_only = [el[TARGET_METRIC] for el in results_with_metric]
        #print(res_metr_only)
        #results_with_metric = results
        #for inner_idx, res in enumerate(results_with_metric):
        #    res[TARGET_METRIC] = metric_scores[inner_idx]
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results)):
            results_with_metric[idx]["estimations"]["UHead_exp1"] = exp1_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp2"] = exp2_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp3"] = exp3_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp4"] = exp4_uhead_scores[idx]
            #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            #    results_with_metric[idx]["estimations"][f"UHead_exp{ue_res_idx+5}"] = el[idx]
            for ue_res_idx, el in enumerate(exp_uhead_scores_array_full_hp):
                results_with_metric[idx]["estimations"][f"UHead_exp_full_hp_{ue_res_idx}"] = el[idx]

        
        '''
        metric_df = pd.read_csv("./med_data/medical_data_ordered_with_claims.csv")
        metric_scores = np.array(metric_df[TARGET_METRIC])[test_ids].flatten()


        def extract_target_metric(results: List[Dict], metric_name: str) -> List[Dict]:
                    valid_results = []
            for res in results:
                if 'estimations' in res and metric_name in res['estimations']:
                    res[metric_name] = res['estimations'][metric_name]
                    valid_results.append(res)
            return valid_results
        #print(results)
        results_with_metric = results
        for inner_idx, res in enumerate(results_with_metric):
            res[TARGET_METRIC] = metric_scores[inner_idx]
        print(results_with_metric[-1])
        '''

        #results_with_metric = extract_target_metric(results, TARGET_METRIC)
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = ['UHead_exp1', 'UHead_exp2', 'UHead_exp3', 'UHead_exp4']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array):
        #    relevant_estimators += [f'UHead_exp{ue_res_idx+5}']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array_full_hp):
            relevant_estimators += [f"UHead_exp_full_hp_{ue_res_idx}"]
        # ['NumSemSets', 'EigValLaplacian_NLI_score_entail',  'EigValLaplacian_NLI_score_contra',
        #                        'EigValLaplacian_NLI_Jaccard_score','DegMat_NLI_score_entail', 'DegMat_NLI_score_contra',
        #                        'DegMat_Jaccard_score', 'Eccentricity_NLI_score_entail', 'Eccentricity_NLI_score_contra',
        #                        'Eccentricity_Jaccard_score', 'LexicalSimilarity_rouge1', 'LUQ',
        #                        'LexicalSimilarity_rouge2', 'LexicalSimilarity_rougeL', 'LexicalSimilarity_BLEU',
        #                        'KernelLanguageEntropy', 'SemanticDensity']
        
        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'./med_ue_res/claim_level_relaxed_em/giga_big/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        prr_05_aggregated_res[name][key] = f'{np.mean(prr_05_aggregated_res[name][key])}±{np.std(prr_05_aggregated_res[name][key])}'
plot_prr_ranking(prr_05_aggregated_res, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_mean.png", "PRR Raw")
plot_prr_ranking(prr_05_aggregated_res, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_mean.png", "PRR Normalized")
plot_prr_ranking(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_mean.png', 'PRR (0.5)')
plot_prr_ranking(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_mean.png', 'PRR Normalized (0.5)')

In [ ]:
# now aggregate scores on k-fold
TARGET_METRIC = 'filtered_claim_is_true'  
OUTPUT_DIR = f'./med_ue_res/claim_level_relaxed_em/giga_big/fixed_format_claim_fix_intro_fix/prr_analysis1_cv_{TARGET_METRIC}'
prr_05_aggregated_res = {}
metric_keys = ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5'] + ['rejection_ratios', 'metric_means']
for kfold_idx in range(len(kfold_test_ids)):
    with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'r') as f:
        kfold_data = json.load(f)
    if kfold_idx == 0:    
        for name in kfold_data.keys():
            prr_05_aggregated_res[name] = {}
            for key in metric_keys:
                prr_05_aggregated_res[name][key] = [kfold_data[name][key]]
    else:
        for name in kfold_data.keys():
            for key in metric_keys:
                prr_05_aggregated_res[name][key].append(kfold_data[name][key])
for name in kfold_data.keys():
    for key in metric_keys:
        if key in ['prr', 'prr_norm', 'prr_0.5', 'prr_norm_0.5']:
            prr_05_aggregated_res[name][key] = np.mean(prr_05_aggregated_res[name][key])
        else:
            prr_05_aggregated_res[name][key] = np.mean(np.array(prr_05_aggregated_res[name][key]), axis=0)

plot_prr_curves(prr_05_aggregated_res, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_mean_no_std.png', 'PRR Raw', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_mean_no_std.png', 'PRR Normalized', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_mean_no_std.png', 'PRR (0.5)', TARGET_METRIC)
plot_prr_curves(prr_05_aggregated_res, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_mean_no_std.png', 'PRR Normalized (0.5)', TARGET_METRIC)


## PRR for k-fold data, relaxed EM, claim-level, Giga big, fixed format, fixed intro, holdout set

In [ ]:
import datasets
base_path_uhead_exp1 = ""
#uhead_exp1_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target/train/2025-11-14/18-30-01/predictions"
#exp1_uhead_scores = datasets.load_from_disk(uhead_exp1_path)
#exp1_uhead_scores = [el[0][0] for el in exp1_uhead_scores['uncertainty_logits']]

In [ ]:
#uhead_exp2_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target_pos_033/train/2025-11-14/19-49-37/predictions"
#exp2_uhead_scores = datasets.load_from_disk(uhead_exp2_path)
#exp2_uhead_scores = [el[0][0] for el in exp2_uhead_scores['uncertainty_logits']]

In [ ]:
#uhead_exp3_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_lite/med_data_hit1_target_attn_5_pos_5/train/2025-11-14/19-55-47/predictions"
#exp3_uhead_scores = datasets.load_from_disk(uhead_exp3_path)
#exp3_uhead_scores = [el[0][0] for el in exp3_uhead_scores['uncertainty_logits']]

In [ ]:
kfold_test_ids = [
    list(range(64)),
]
kfold_test_pathes = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim/train/2025-12-24/12-02-15/predictions/",
]

kfold_test_pathes_exp2 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_pos_033/train/2025-12-24/12-02-58/predictions/",
]

kfold_test_pathes_exp3 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_attn_5_pos_5/train/2025-12-24/12-03-47/predictions/",
]

kfold_test_pathes_exp4 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/12-04-28/predictions/",
]


# now add enlarged train (with same test splits)
enlarged_train_kfold_test_pathes_exp1 = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_hp_prev/train/2025-12-24/12-05-15/predictions/",
]




kfold_test_pathes_exp1_full_train = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_full_train/train/2025-12-24/12-05-54/predictions/",
]

kfold_test_pathes_exp2_full_train = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_pos_033_full_train/train/2025-12-24/12-06-35/predictions/",
]

kfold_test_pathes_exp3_full_train = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_attn_5_pos_5_full_train/train/2025-12-24/12-07-17/predictions/",
]

kfold_test_pathes_exp4_full_train = [
    "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_pos_1_full_train/train/2025-12-24/12-09-06/predictions/",
]

In [ ]:
x=1

In [ ]:
import pandas as pd
import copy

In [ ]:
def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key[12:]] = df.iloc[idx][key]
        results.append(buf_res)
    return results

In [ ]:
#kfold_test_ids

In [ ]:
def main():
    log_file = '/home/jovyan/rvanova/polygraph_nemo_repos/lm-polygraph/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    TARGET_METRIC = 'filtered_claim_is_true'  
    OUTPUT_DIR = f'./med_ue_res/claim_level_relaxed_em/giga_big/fixed_format_claim_fix_intro_fix/holdout_test/prr_analysis1_cv_{TARGET_METRIC}'
    
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    for kfold_idx in range(len(kfold_test_ids)):
        test_ids = kfold_test_ids[kfold_idx]
        uhead_exp1_path = kfold_test_pathes[kfold_idx]
        exp1_uhead_scores = datasets.load_from_disk(uhead_exp1_path)
        exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
        exp1_uhead_scores = exp1_uhead_scores[exp1_uhead_scores != -100]
        uhead_exp2_path = kfold_test_pathes_exp2[kfold_idx]
        exp2_uhead_scores = datasets.load_from_disk(uhead_exp2_path)
        exp2_uhead_scores = np.array(exp2_uhead_scores['uncertainty_logits'])
        exp2_uhead_scores = exp2_uhead_scores[exp2_uhead_scores != -100]
        uhead_exp3_path = kfold_test_pathes_exp3[kfold_idx]
        exp3_uhead_scores = datasets.load_from_disk(uhead_exp3_path)
        exp3_uhead_scores = np.array(exp3_uhead_scores['uncertainty_logits'])
        exp3_uhead_scores = exp3_uhead_scores[exp3_uhead_scores != -100]
        uhead_exp4_path = kfold_test_pathes_exp4[kfold_idx]
        exp4_uhead_scores = datasets.load_from_disk(uhead_exp4_path)
        exp4_uhead_scores = np.array(exp4_uhead_scores['uncertainty_logits'])
        exp4_uhead_scores = exp4_uhead_scores[exp4_uhead_scores != -100]

        exp_uhead_scores_array = []
        for exp_path in [enlarged_train_kfold_test_pathes_exp1, kfold_test_pathes_exp1_full_train,
                         kfold_test_pathes_exp2_full_train, kfold_test_pathes_exp3_full_train,
                         kfold_test_pathes_exp4_full_train]:
            uhead_exp_buf_path = exp_path[kfold_idx]
            exp_buf_uhead_scores = datasets.load_from_disk(uhead_exp_buf_path)
            exp_buf_uhead_scores = np.array(exp_buf_uhead_scores['uncertainty_logits'])
            exp_buf_uhead_scores = exp_buf_uhead_scores[exp_buf_uhead_scores != -100]
            exp_uhead_scores_array.append(copy.deepcopy(exp_buf_uhead_scores))

        
        path = "./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv"
        df = pd.read_csv(path)
        results = results_from_csv(df)#parse_log_file(log_content, medical_matcher)
        results = [results[idx] for idx in test_ids]
        # for claim level - 
        metric_df = pd.read_csv("./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv")
        #metric_scores = list(metric_df[TARGET_METRIC])
        #print(metric_scores)
        metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]
        #metric_scores = [[int(1-x) for x in el] for el in metric_scores]
        #metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(".","").split())) for idx in test_ids]
        # now duplicate all samples wrt claims num
        print(metric_scores)
        print(len(results))
        results_with_metric = []
        for inner_idx, el in enumerate(metric_scores):
            for inner_el in el:
                buf_res = copy.deepcopy(results[inner_idx])
                buf_res[TARGET_METRIC] = inner_el
                results_with_metric.append(buf_res)
                #results_with_metric[-1][TARGET_METRIC] = inner_el
        print(len(results_with_metric), len(exp2_uhead_scores), sum([len(x) for x in metric_scores]))
        #res_metr_only = [el[TARGET_METRIC] for el in results_with_metric]
        #print(res_metr_only)
        #results_with_metric = results
        #for inner_idx, res in enumerate(results_with_metric):
        #    res[TARGET_METRIC] = metric_scores[inner_idx]
        print(results_with_metric[-1])
        flat_metrics = [el["filtered_claim_is_true"] for el in results_with_metric]
        print(np.unique(flat_metrics, return_counts=True), len(flat_metrics))
        print(np.unique(flat_metrics, return_counts=True)[1][1] / np.sum(np.unique(flat_metrics, return_counts=True)[1]))
        
        
        # insert uhead scores
        for idx in range(len(results)):
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_0"] = exp1_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_1"] = exp2_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_2"] = exp3_uhead_scores[idx]
            results_with_metric[idx]["estimations"]["UHead_exp_full_hp_3"] = exp4_uhead_scores[idx]
            for ue_res_idx, el in enumerate(exp_uhead_scores_array):
                if ue_res_idx == 0:
                    results_with_metric[idx]["estimations"][f"UHead_exp_full_hp_{ue_res_idx+4}"] = el[idx]
                else:
                    results_with_metric[idx]["estimations"][f"UHead_exp_{ue_res_idx}_full_train"] = el[idx]
            #for ue_res_idx, el in enumerate(exp_uhead_scores_array_full_hp):
            #    results_with_metric[idx]["estimations"][f"UHead_exp_full_hp_{ue_res_idx}"] = el[idx]

        
        '''
        metric_df = pd.read_csv("./med_data/medical_data_ordered_with_claims.csv")
        metric_scores = np.array(metric_df[TARGET_METRIC])[test_ids].flatten()


        def extract_target_metric(results: List[Dict], metric_name: str) -> List[Dict]:
                    valid_results = []
            for res in results:
                if 'estimations' in res and metric_name in res['estimations']:
                    res[metric_name] = res['estimations'][metric_name]
                    valid_results.append(res)
            return valid_results
        #print(results)
        results_with_metric = results
        for inner_idx, res in enumerate(results_with_metric):
            res[TARGET_METRIC] = metric_scores[inner_idx]
        print(results_with_metric[-1])
        '''

        #results_with_metric = extract_target_metric(results, TARGET_METRIC)
        
        if not results_with_metric:
            print(f"No valid results found with {TARGET_METRIC}.")
            return
        
        print(f"Found {len(results_with_metric)} results with {TARGET_METRIC}")
        
        all_estimators = set()
        for res in results_with_metric:
            if 'estimations' in res:
                all_estimators.update(res['estimations'].keys())
        
        relevant_estimators = ['DegMat_NLI_score_contra', 'LexicalSimilarity_rouge2', 'PPLRMDSeq_decoder', 'MaximumSequenceProbability', 
         'DegMat_NLI_score_entail', 'DegMat_Jaccard_score', 'RDESeq_decoder', 'MonteCarloSequenceEntropy', 
         'MeanPointwiseMutualInformation', 'PPLMDSeq_decoder', 'MahalanobisDistanceSeq_decoder', 'AttentionScore (layer=17)', 
         'SemanticDensity', 'SelfCertainty', 'LexicalSimilarity_rougeL', 'LUQ', 'CCP', 
         'RelativeMahalanobisDistanceSeq_decoder', 'CSL', 'MeanTokenEntropy', 'EigValLaplacian_NLI_score_entail', 'CocoaPPL', 
         'PTrueSampling', 'EigenScore', 'CocoaMTE', 'LexicalSimilarity_BLEU', 'CocoaMSP', 
         'LexicalSimilarity_rouge1', 'RAUQ', 'PTrue', 'NumSemSets', 'MeanConditionalPointwiseMutualInformation', 'RAUQ (entropy)', 
         'MonteCarloNormalizedSequenceEntropy', 'KernelLanguageEntropy', 'SemanticEntropy']
        
        relevant_estimators = ['UHead_exp_full_hp_0', 'UHead_exp_full_hp_1', 'UHead_exp_full_hp_2', 'UHead_exp_full_hp_3']#, 'UHead_exp5', 'UHead_exp6']#, 'UHead_exp7']
        for ue_res_idx, el in enumerate(exp_uhead_scores_array):
            if ue_res_idx == 0:
                relevant_estimators += [f'UHead_exp_full_hp_{ue_res_idx+4}']
            else:
                relevant_estimators += [f'UHead_exp_{ue_res_idx}_full_train']
        #for ue_res_idx, el in enumerate(exp_uhead_scores_array_full_hp):
        #    relevant_estimators += [f"UHead_exp_full_hp_{ue_res_idx}"]
        # ['NumSemSets', 'EigValLaplacian_NLI_score_entail',  'EigValLaplacian_NLI_score_contra',
        #                        'EigValLaplacian_NLI_Jaccard_score','DegMat_NLI_score_entail', 'DegMat_NLI_score_contra',
        #                        'DegMat_Jaccard_score', 'Eccentricity_NLI_score_entail', 'Eccentricity_NLI_score_contra',
        #                        'Eccentricity_Jaccard_score', 'LexicalSimilarity_rouge1', 'LUQ',
        #                        'LexicalSimilarity_rouge2', 'LexicalSimilarity_rougeL', 'LexicalSimilarity_BLEU',
        #                        'KernelLanguageEntropy', 'SemanticDensity']
        
        print(relevant_estimators)
        
        print(f"Available estimators ({len(relevant_estimators)}): {relevant_estimators}")
        
        prr_results = {}
        all_target_scores = []
        
        for estimator in relevant_estimators:
            uncertainties, target_scores = [], []
            
            for res in results_with_metric:
                if (estimator in res['estimations'] and 
                    TARGET_METRIC in res):
                    uncertainties.append(res['estimations'][estimator])
                    target_scores.append(res[TARGET_METRIC])
            
            if uncertainties and len(uncertainties) == len(target_scores):
                all_target_scores = target_scores
                
                prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, target_scores, 1.0)
                prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, target_scores, 0.5)
                
                sort_idx = np.argsort(uncertainties)
                sorted_scores = np.array(target_scores)[sort_idx]
                rejection_ratios = np.linspace(0, 1.0, 95)
                metric_means = [
                    np.mean(sorted_scores[:int(len(sorted_scores)*(1-r))])
                    if int(len(sorted_scores)*(1-r)) > 0 else 0.0
                    for r in rejection_ratios
                ]
                
                prr_results[estimator] = {
                    'rejection_ratios': rejection_ratios,
                    'metric_means': metric_means,
                    'prr': prr_raw,
                    'prr_norm': prr_norm,
                    'prr_0.5': prr05_raw,
                    'prr_norm_0.5': prr05_norm,
                    'target_scores_raw': target_scores,
                    'uncertainties_raw': uncertainties
                }
                print(f"✓ {estimator}: PRR={prr_raw:.4f}, norm={prr_norm:.4f}")
            else:
                print(f"✗ {estimator}: No valid data")
        
        if all_target_scores:
            oracle_ue = 1.0 - np.array(all_target_scores)
            rejection_ratios, oracle_curve = compute_curve(oracle_ue, all_target_scores)
        
            prr_results['Oracle'] = {
                'rejection_ratios': rejection_ratios,
                'metric_means': oracle_curve,
                'prr': calculate_prr_area(oracle_ue, all_target_scores, 1.0),
                'prr_norm': 1.0,
                'prr_0.5': calculate_prr_area(oracle_ue, all_target_scores, 0.5),
                'prr_norm_0.5': 1.0,
            }
            print("✓ Oracle: added")
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        print(f"\nPRR results keys: {list(prr_results.keys())}")
        
        if len(prr_results) > 1:
            print(f"\nBuilding plots for {len(prr_results) - 1} estimators...")
            
            plot_prr_curves(prr_results, 'prr', f'{OUTPUT_DIR}/prr_curves_cv_{kfold_idx}.png', 'PRR Raw', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm', f'{OUTPUT_DIR}/prr_norm_curves_cv_{kfold_idx}.png', 'PRR Normalized', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/prr05_curves_cv_{kfold_idx}.png', 'PRR (0.5)', TARGET_METRIC)
            plot_prr_curves(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/prr_norm05_curves_cv_{kfold_idx}.png', 'PRR Normalized (0.5)', TARGET_METRIC)
            
            plot_prr_ranking(prr_results, "prr", f"{OUTPUT_DIR}/ranking_prr_cv_{kfold_idx}.png", "PRR Raw")
            plot_prr_ranking(prr_results, "prr_norm", f"{OUTPUT_DIR}/ranking_prr_norm_cv_{kfold_idx}.png", "PRR Normalized")
            plot_prr_ranking(prr_results, 'prr_0.5', f'{OUTPUT_DIR}/ranking_prr_05_cv_{kfold_idx}.png', 'PRR (0.5)')
            plot_prr_ranking(prr_results, 'prr_norm_0.5', f'{OUTPUT_DIR}/ranking_prr_norm_05_cv_{kfold_idx}.png', 'PRR Normalized (0.5)')
            
            with open(f"{OUTPUT_DIR}/prr_results_cv_{kfold_idx}.json", 'w') as f:
                json_ready = {}
                for name, data in prr_results.items():
                    json_ready[name] = {
                        'rejection_ratios': [float(x) for x in data['rejection_ratios']],
                        'metric_means': [float(x) for x in data['metric_means']],
                        'prr': float(data['prr']),
                        'prr_norm': float(data['prr_norm']),
                        'prr_0.5': float(data['prr_0.5']),
                        'prr_norm_0.5': float(data['prr_norm_0.5'])
                    }
                json.dump(json_ready, f, indent=4)
            
            print(f"\nAll results saved to: {OUTPUT_DIR}/")
        else:
            print("Not enough data to build plots")
        
        print("\nPRR analysis completed!")

if __name__ == "__main__":
    main()

### Now estimate optimal threshold and manually check results

In [ ]:
from scipy.special import expit
import numpy as np
import datasets
import pandas as pd

In [ ]:
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format_claim_fix_intro_fix/holdout_evals/med_data_relaxed_em_target_claim_pos_1/train/2025-12-24/12-04-28/predictions/"
TARGET_METRIC = 'filtered_claim_is_true'
exp1_uhead_scores = datasets.load_from_disk(preds_path)
exp1_uhead_scores_raw = np.array(exp1_uhead_scores['uncertainty_logits'])
exp1_uhead_scores = expit(exp1_uhead_scores_raw)#expit(exp1_uhead_scores[exp1_uhead_scores != -100])
exp1_uhead_scores_flat = expit(exp1_uhead_scores_raw[exp1_uhead_scores_raw != -100])
path = "./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv"
df = pd.read_csv(path)
results = results_from_csv(df)
test_ids = list(range(len(df)))
results = [results[idx] for idx in test_ids]
# for claim level - 
metric_df = pd.read_csv("./med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims_neural_intro_fix.csv")
metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay, roc_curve

In [ ]:
metric_scores_flat = []
for el in metric_scores:
    metric_scores_flat += el

In [ ]:
precision, recall, thresholds = precision_recall_curve(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
PrecisionRecallDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
thresholds[-1]

In [ ]:
RocCurveDisplay.from_predictions(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
fpr, tpr, thr = roc_curve(metric_scores_flat, 1-exp1_uhead_scores_flat)

In [ ]:
pareto_score = np.sqrt((1-tpr)**2 + (fpr)**2)
opt_point = np.min(pareto_score)
opt_thr = thr[np.argmin(pareto_score)]
opt_point, opt_thr

In [ ]:
pareto_score = np.sqrt((1-tpr)**2)
opt_point = np.min(pareto_score)
opt_thr = thr[np.argmin(pareto_score)]
opt_point, opt_thr

In [ ]:
1-exp1_uhead_scores[0]

In [ ]:
gt_ids = np.arange(len(metric_scores_flat))
gt_ids = [gt_ids[idx] for idx, el in enumerate(metric_scores_flat) if el==1]
false_ids = np.arange(len(metric_scores_flat))
false_ids = [false_ids[idx] for idx, el in enumerate(metric_scores_flat) if el==0]
gt_ues = (1-exp1_uhead_scores_flat)[gt_ids]
false_ues = (1-exp1_uhead_scores_flat)[false_ids]

In [ ]:
np.min(gt_ues), np.max(gt_ues), np.median(gt_ues), np.percentile(gt_ues, 90), np.percentile(gt_ues, 5)

In [ ]:
np.min(false_ues), np.max(false_ues), np.median(false_ues), np.percentile(false_ues, 95), np.percentile(false_ues, 10)

In [ ]:
metric_scores_flat

In [ ]:
thr

In [ ]:
#pareto_score

In [ ]:
prob_idx = -14
best_thr = 0.3770#0.3083
high_thr = 0.4765#0.8
print(results[prob_idx]["input"])
print("Pred:", results[prob_idx]["prediction"])
print("Target:", results[prob_idx]["target"])
print(results[prob_idx]["estimations"])

print(1-exp1_uhead_scores[prob_idx])

In [ ]:
results[prob_idx]["input"][43:46]

In [ ]:
male_stats = []
female_stats = []
for idx in range(len(results)):
    if "мужского" in results[idx]["input"]:
        male_stats.append(results[idx]["estimations"]["ue"])
    else:
        female_stats.append(results[idx]["estimations"]["ue"])

In [ ]:
male_scores = [list(map(float, male_stats[idx].replace("[","").replace("]","").replace(",","").split())) for idx in range(len(male_stats))]
female_scores = [list(map(float, female_stats[idx].replace("[","").replace("]","").replace(",","").split())) for idx in range(len(female_stats))]

In [ ]:
male_scores_flat = []
for el in male_scores:
    male_scores_flat += el
female_scores_flat = []
for el in female_scores:
    female_scores_flat += el

In [ ]:
np.mean(male_scores_flat), len(male_scores_flat), np.mean(female_scores_flat), len(female_scores_flat)

In [ ]:
results[0]

In [ ]:
exp1_uhead_scores

## Now prepare several mock samples

In [ ]:
mocks = [
    """Ты врач с большим опытом работы.\nЯ пациент мужского пола. Мне 45 лет. Я обратился к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> На что жалуетесь?
<пациент> Давление скачет, и голова от этого раскалывается.
<врач> Как давно это беспокоит?
<пациент> Лет пять уже, но последний месяц совсем плохо, почти каждый день.
<врач> Какие цифры давления обычно видите?
<пациент> Поднимается до 170 на 100, бывает и выше.
-<врач> Принимаете что-то постоянно?
<пациент> Пью анаприлин, когда прижмет, но помогает ненадолго.
<врач> Есть ли головокружение, мушки перед глазами?
<пациент> Да, особенно когда резко встаю, в глазах темнеет.
<врач> Беспокоит ли боль в груди, одышка?
<пациент> Одышка есть, когда по лестнице поднимаюсь. Боли в груди нет.
<врач> Курите, употребляете алкоголь?
<пациент> Курил, но бросил год назад. Алкоголь – по праздникам.
<врач> Есть ли у родственников проблемы с давлением, сердцем?
<пациент> Отец умер от инфаркта, у матери гипертония.
<пациент> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациент> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент мужского пола. Мне 25 лет. Я обратился к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> На что жалуетесь?
<пациент> Сильно болит горло, глотать почти невозможно.
<врач> Как давно началось?
<пациент> Два дня назад.
<врач> Температура есть?
<пациент> Да, вчера была 38.5, сегодня 37.8.
<врач> Что-то еще беспокоит, кроме боли в горле?
<пациент> Слабость сильная, голова болит, ломит все тело.
<врач> Можете открыть рот и показать горло? Есть ли налет на миндалинах?
<пациент> Да, там какие-то белые точки.
<врач> Увеличились ли лимфоузлы на шее?
<пациент> Да, с правой стороны болезненная шишка.
<врач> Был ли контакт с больными?
<пациент> На работе несколько человек болели.
<пациент> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациент> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент мужского пола. Мне 32 лет. Я обратился к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> На что жалуетесь?
<пациент> Постоянно чувствую себя разбитым, нет сил вообще.
<врач> Как давно это продолжается?
<пациент> Уже несколько месяцев, но последние две недели совсем плохо.
<врач> Расскажите подробнее о своем состоянии. Что еще беспокоит?
<пациент> Постоянно хочется спать, даже если сплю 8-9 часов. Просыпаюсь уже уставшим. И появилась апатия, ничего не хочется.
<врач> Были ли стрессовые ситуации, повышенные нагрузки на работе?
<пациент> Работа очень нервная, проект авральный уже полгода. Домой прихожу выжатый как лимон.
<врач> Изменился ли аппетит, вес?
<пациент> Аппетит плохой, похудел килограмма на три за последний месяц.
<врач> Замечали ли вы проблемы с концентрацией внимания, памятью?
<пациент> Да, голова как в тумане, ничего не соображаю, забываю элементарные вещи.
<врач> Есть ли физические симптомы? Головные боли, например?
<пациент> Голова болит нечасто, но постоянно ощущение слабости в теле, будто ватное.
<врач> Обращались ли вы с такими жалобами к другим врачам, сдавали анализы?
<пациент> Нет, думал, само пройдет, но только хуже становится.
<пациент> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациент> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент мужского пола. Мне 57 лет. Я обратился к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> Что вас беспокоит?
<пациент> Очень сильно болит поясница, отдает в правую ногу.
<врач> Как давно это началось?
<пациент> Неделю назад прихватило после того, как поднял тяжелый ящик.
<врач> Боль постоянная или усиливается при движении?
<пациент> Постоянная, ноющая. Когда хожу или наклоняюсь — становится стреляющей и резкой.
<врач> Было ли онемение или слабость в ноге, пальцах?
<пациент> Да, по внешней стороне бедра и до пальцев немеет, чувство мурашек.
<врач> Что пробовали делать для облегчения?
<пациент> Мазал обезболивающей мазью и старался лежать, но лучше не становится.
<врач> Были ли подобные проблемы раньше?
<пациент> Да, года три назад прострелы были, но так сильно никогда не болело.
<врач> Есть ли у вас хронические заболевания, например, сахарный диабет?
<пациент> Нет, в целом здоров, только давление иногда повышается.
<пациент> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациент> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент мужского пола. Мне 63 лет. Я обратился к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> Что вас беспокоит?
<пациент> Слабость в правой руке и ноге, и речь немного заплетается.
<врач> Как давно это появилось?
<пациент> Сегодня утром, когда встал с кровати.
<врач> Была ли потеря сознания, головокружение?
<пациент> Нет, сознание не терял. Голова немного кружилась.
<врач> Есть ли онемение в этих конечностях, асимметрия лица?
<пациент> Рука и нога как будто чужие, плохо слушаются. Лицо вроде нормальное.
<врач> Повышается ли у вас артериальное давление обычно?
<пациент> Да, гипертония давно, таблетки пью нерегулярно.
<врач> Болели ли вы сахарным диабетом, были ли проблемы с сердцем?
<пациент> Диабета нет. Сердце пошаливает, аритмия.
<врач> Курите ли вы?
<пациент> Бросил 10 лет назад, до этого курил долго.
<пациент> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациент> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 63 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> Что вас беспокоит?
<пациентка> Сильно болит и опухло колено, на ногу наступить не могу.
<врач> Какое именно колено и как давно?
<пациентка> Правое. Вчера вечером началось, а сегодня утром уже почти не гнется.
<врач> Была ли травма, падение?
<пациентка> Нет, ничего такого не было. Просто встала со стула, и как будто стрельнуло.
<врач> Были ли у вас раньше подобные приступы, может, в других суставах?
<пациентка> Да, года два назад палец на ноге так распухал и болел страшно. Тогда сказали, что подагра.
<врач> Повышалась ли температура?
<пациентка> Немного, 37.3 чувствую озноб.
<врач> Колено на ощупь горячее, кожа красная?
<пациентка> Да, прямо горит и покраснело.
<врач> Принимали ли вы что-то от боли?
<пациентка> Выпила парацетамол, но почти не помогло.
<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 55 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> Что вас беспокоит?
<пациентка> Постоянно кидает то в жар, то в холод, и настроение ужасное.
<врач> Как давно это началось?
<пациентка> Примерно полгода, но последний месяц особенно тяжело.
<врач> Расскажите подробнее о приливах жара.
<пациентка> Внезапно становится очень жарко, лицо и шея краснеют, сильно потею. Длится минуты две-три, потом проходит и иногда даже знобит.
<врач> Нарушен ли сон?
<пациентка> Да, ночью просыпаюсь от этих приливов, а потом долго не могу заснуть. А днем разбитая.
<врач> Изменился ли менструальный цикл?
<пациентка> Месячные стали очень нерегулярными, с большими задержками, а в прошлом месяце вообще не было.
<врач> Чувствуете ли вы сухость слизистых, например, во влагалище?
<пациентка> Да, есть такая проблема, и дискомфорт.
<врач> Беспокоят ли перепады настроения, раздражительность?
<пациентка> Постоянно! Плачу по пустякам, на всех срываюсь, ничего не радует. Сама себе не нравлюсь.
<врач> Есть ли у вас хронические заболевания, операции?
<пациентка> Нет, в целом здорова, только аппендицит в детстве удаляли.
<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 25 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> На что жалуетесь?
<пациентка> Сильно кружится голова, все плывет перед глазами.
<врач> Как давно это началось?
<пациентка> Вчера вечером первый раз, а сегодня утром повторилось, когда встала с кровати.
<врач> Опишите ощущения подробнее. Это чувство вращения или неустойчивости?
<пациентка> Кажется, что комната крутится вокруг меня. Тошнит от этого.
<врач> Случалась ли рвота?
<пациентка> Да, один раз сегодня утром.
<врач> Ухудшается ли головокружение при повороте головы или изменении положения тела?
<пациентка> Да! Особенно когда ложусь или поворачиваюсь на бок в кровати. Все начинает сильно кружиться.
<врач> Слышите ли вы шум или звон в ушах, есть ли чувство заложенности?
<пациентка> В правом ухе шумит постоянно, и как будто немного заложило.
<врач> Были ли у вас простудные заболевания, отиты в последнее время?
<пациентка> Неделю назад сильно болело горло и был насморк.
<врач> Травм головы или шеи не было?
<пациентка> Нет.
<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 32 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> На что жалуетесь?
<пациентка> Сильная боль в низу живота, больше справа.
<врач> Как давно появилась боль?
<пациентка> Сегодня днем началось, постепенно усиливается.
<врач> Боль постоянная или схваткообразная?
<пациентка> Постоянная, ноющая, отдает в поясницу.
<врач> Повышалась ли температура?
<пациентка> Да, 37.8.
<врач> Была ли тошнота или рвота?
<пациентка> Тошнило, один раз вырвало.
<врач> Изменился ли характер выделений из влагалища?
<пациентка> Да, с утра заметила более обильные желтоватые выделения с неприятным запахом.
<врач> Когда у вас были последние месячные? Регулярный ли цикл?
<пациентка> Месячные закончились неделю назад. Цикл регулярный.
<врач> Есть ли боль или жжение при мочеиспускании?
<пациентка> Да, немного больно.
<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
    """Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 45 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>
<врач> Что вас беспокоит?
<пациентка> Ужасно болит голова, прямо раскалывается.
<врач> Опишите боль. Где именно болит?
<пациентка> В правой половине головы, в области виска и глаза. Пульсирующая, очень сильная.
<врач> Как давно начался приступ?
<пациентка> Около часа назад. Сначала стало мелькать в глазах, а потом началась боль.
-<врач> Была ли тошнота или рвота?
<пациентка> Да, тошнило, один раз вырвало. Свет и звук сейчас невыносимы.
<врач> Были ли у вас раньше такие приступы?
<пациентка> Да, периодически, но не так сильно. Обычно раз в несколько месяцев.
<врач> Что-то провоцирует начало боли? Стресс, определенная еда, недосып?
<пациентка> Иногда начинается после недосыпа или сильного волнения. Сегодня как раз был тяжелый день.
<врач> Принимали ли вы что-то от боли?
<пациентка> Выпила ибупрофен, но не помогло.
<врач> Есть ли у вас хронические заболевания, проблемы с давлением?
<пациентка> Давление обычно в норме. Миопия средней степени.
<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском
<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз
</диалог>
Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами
Каждый ответ оберни в теги <diag_diag> и </diag_diag>""",
]

In [ ]:
mock_df = pd.DataFrame()
mock_df["input"] = mocks
mock_path = "./med_data/mock_samples.csv"
mock_df.to_csv(mock_path, index=False)

In [ ]:
45, 25, 32, 57, 63

In [ ]:
mock_dict = {"prompts": [], "outputs": [], "claims": [], "use_threshold": []}
input_keys = ["prompts", "outputs", "claims", "use_threshold"]
if not(all([key in mock_dict.keys() for key in input_keys])):
    print(400)

## Now try to run pretrained model on sample

## Some other res

In [ ]:
#norma 4 graphics ORACLE
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
import os

def extract_alignscores(results: List[Dict]) -> List[Dict]:
    valid_results = []
    for res in results:
        if 'estimations' in res and 'max_matching_score' in res['estimations']:
            
            res['max_matching_score'] = res['estimations']['max_matching_score']
            valid_results.append(res)
    return valid_results

# --------------------------
# Normalization helpers
# --------------------------
def normalize(target: List[float]):
    min_t, max_t = np.min(target), np.max(target)
    if np.isclose(min_t, max_t):
        min_t -= 1
        max_t += 1
    target = (np.array(target) - min_t) / (max_t - min_t)
    return target

def get_random_scores(function, metrics, num_iter=200, seed=42):
    np.random.seed(seed)
    rand_scores = np.arange(len(metrics))
    value = []
    for _ in range(num_iter):
        np.random.shuffle(rand_scores)
        rand_val = function(rand_scores, metrics)
        value.append(rand_val)
    return np.mean(value)

def normalize_metric(target_score, oracle_score, random_score):
    if not (oracle_score == random_score):
        target_score = (target_score - random_score) / (oracle_score - random_score)
    return target_score

# --------------------------
# Core PRR calc
# --------------------------
def calculate_prr_area(estimator: List[float], target: List[float], max_rejection: float = 1.0) -> float:
    target = normalize(target)
    ue = np.array(estimator)
    num_obs = len(ue)
    num_rej = int(max_rejection * num_obs)

    ue_argsort = np.argsort(ue)
    sorted_metrics = np.array(target)[ue_argsort]
    cumsum = np.cumsum(sorted_metrics)[-num_rej:]
    scores = (cumsum / np.arange((num_obs - num_rej) + 1, num_obs + 1))[::-1]
    prr_score = np.sum(scores) / num_rej
    return prr_score

def compute_prr_and_normalized(estimator: List[float], target: List[float], max_rejection: float = 1.0):
    raw_score = calculate_prr_area(estimator, target, max_rejection)
    oracle_ue = 1.0 - normalize(target)
    oracle_score = calculate_prr_area(oracle_ue, target, max_rejection)
    random_score = get_random_scores(
        lambda e, t: calculate_prr_area(e, t, max_rejection), target
    )
    norm_score = normalize_metric(raw_score, oracle_score, random_score)
    return raw_score, norm_score

def compute_curve(estimator, target, max_rejection=1.0, num_points=95, do_normalize=False):
    



    target = normalize(target) if do_normalize else np.asarray(target)
    ue = np.asarray(estimator)
    num_obs = len(ue)

    
    sort_idx = np.argsort(ue)
    sorted_metrics = target[sort_idx]

    rejection_ratios = np.linspace(0, max_rejection, num_points)
    alignscore_means = []
    for rr in rejection_ratios:
        n_keep = int(num_obs * (1 - rr))
        alignscore_means.append(np.mean(sorted_metrics[:n_keep]) if n_keep > 0 else 0.0)
    return rejection_ratios, alignscore_means

# --------------------------
# Plotting functions
# --------------------------
def plot_metric_with_oracle(prr_results, key, output_file, ylabel, top_n=8):
    


    plt.figure(figsize=(12, 8))
    
    
    sorted_items = sorted(
        [(name, vals) for name, vals in prr_results.items() if name != "Oracle" and key in vals],
        key=lambda x: x[1][key],
        reverse=True
    )
    if not sorted_items:
        print(f"[warn] nothing to plot for key={key}")
        return
    
    top_estimators = [name for name, _ in sorted_items[:top_n]]
    colors = plt.cm.Set3(np.linspace(0, 1, min(top_n, len(top_estimators))))
    
    
    for i, name in enumerate(top_estimators):
        vals = prr_results[name]
        plt.plot(vals['rejection_ratios'], vals['alignscore_means'],
                 '-', linewidth=3, color=colors[i],
                 label=f"{name} ({key}={vals[key]:.3f})", alpha=0.85)
    
    
    anchor = top_estimators[0]
    anchor_vals = prr_results[anchor]
    
    
    oracle_ue = 1.0 - np.asarray(anchor_vals['alignscores_raw'])
    rr_oracle, curve_oracle = compute_curve(
        oracle_ue,
        anchor_vals['alignscores_raw'],
        max_rejection=1.0,
        num_points=len(anchor_vals['rejection_ratios']),
        do_normalize=False
    )
    
    plt.plot(rr_oracle, curve_oracle,
             '-', linewidth=3.5, color='black',
             label=f"Oracle ({key}={prr_results['Oracle'][key]:.3f})")
    
    plt.title(f'PRR Curves - {ylabel}', fontsize=14, fontweight='bold')
    plt.xlabel('Rejection Ratio', fontsize=12)
    plt.ylabel('Mean Min Matching Score', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 0.9)
    
    
    all_means = []
    for name, vals in prr_results.items():
        if name != "Oracle" and 'alignscore_means' in vals:
            all_means.extend(vals['alignscore_means'])
    
    if all_means:
        y_min = max(0.0, np.min(all_means) - 0.05)
        y_max = min(1.0, np.max(all_means) + 0.05)
        plt.ylim(y_min, y_max)
    else:
        plt.ylim(0.0, 1.0)
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()
    print(f"Saved plot: {output_file}")

def plot_prr_ranking(prr_results: Dict[str, Dict], key: str, output_file: str, ylabel: str, top_n: int = 20):
    sorted_estimators = sorted(prr_results.items(), key=lambda x: x[1][key], reverse=True)
    top_estimators = sorted_estimators[:top_n]

    names = [name for name, _ in top_estimators]
    values = [data[key] for _, data in top_estimators]

    plt.figure(figsize=(10, 6))
    bars = plt.barh(range(len(names)), values, color=plt.cm.Set3(np.linspace(0, 1, len(names))))
    plt.yticks(range(len(names)), names, fontsize=9)
    plt.xlabel(ylabel, fontsize=12)
    plt.title(f"Top-{top_n} estimators by {ylabel}", fontsize=14, fontweight="bold")
    plt.gca().invert_yaxis()  

    for i, bar in enumerate(bars):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{values[i]:.3f}", va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

# --------------------------
# Main
# --------------------------
def main():
    log_file = '/home/jovyan/rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    results = parse_log_file(log_content)
    results_with_alignscores = extract_alignscores(results)
    
    if not results_with_alignscores:
        print("No valid results found with max_matching_score.")
        return

    print(f"Found {len(results_with_alignscores)} results with max_matching_score")
    
    
    all_estimators = set()
    for res in results_with_alignscores:
        if 'estimations' in res:
            all_estimators.update(res['estimations'].keys())
    
    
    relevant_estimators =  [
        'MaximumSequenceProbability', 'Perplexity', 'MeanTokenEntropy',
        'MeanPointwiseMutualInformation', 'SelfCertainty',
        'MeanConditionalPointwiseMutualInformation', 'CCP', 'PTrue',
        'PTrueSampling', 'MonteCarloSequenceEntropy',
        'MonteCarloNormalizedSequenceEntropy', 'LexicalSimilarity_rouge1',
        'LexicalSimilarity_rouge2', 'LexicalSimilarity_rougeL',
        'LexicalSimilarity_BLEU', 'NumSemSets', 'EigValLaplacian_NLI_score_entail',
        'DegMat_NLI_score_entail', 'DegMat_NLI_score_contra', 'DegMat_Jaccard_score',
        'SemanticEntropy', 'LUQ', 'KernelLanguageEntropy', 'EigenScore',
        'MahalanobisDistanceSeq_decoder', 'RelativeMahalanobisDistanceSeq_decoder',
        'RDESeq_decoder', 'PPLMDSeq_decoder', 'PPLRMDSeq_decoder',
        'AttentionScore (layer=14)', 'RAUQ', 'RAUQ (entropy)', 'CSL',
        'CocoaMSP', 'CocoaPPL', 'CocoaMTE', 'SemanticDensity'
    ]
    
    print(f"Available estimators: {relevant_estimators}")
    
    prr_results = {}
    alignscores = []  
    
    for estimator in relevant_estimators:
        uncertainties, matching_scores = [], []
        for res in results_with_alignscores:
            if estimator in res['estimations'] and 'max_matching_score' in res:
                uncertainties.append(res['estimations'][estimator])
                matching_scores.append(res['max_matching_score'])

        if uncertainties and len(uncertainties) == len(matching_scores):
            alignscores = matching_scores  
            prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, matching_scores, 1.0)
            prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, matching_scores, 0.5)

            sort_idx = np.argsort(uncertainties)
            sorted_matching_scores = np.array(matching_scores)[sort_idx]
            rejection_ratios = np.linspace(0, 1.0, 95)
            alignscore_means = [
                np.mean(sorted_matching_scores[:int(len(sorted_matching_scores)*(1-r))])
                if int(len(sorted_matching_scores)*(1-r)) > 0 else 0
                for r in rejection_ratios
            ]

            prr_results[estimator] = {
                'rejection_ratios': rejection_ratios.tolist(),
                'alignscore_means': alignscore_means,
                'prr': prr_raw,
                'prr_norm': prr_norm,
                'prr_0.5': prr05_raw,
                'prr_norm_0.5': prr05_norm,
                'alignscores_raw': matching_scores,      
                'uncertainties_raw': uncertainties       
            }
            print(f"{estimator}: prr={prr_raw:.4f}, norm={prr_norm:.4f}, "
                  f"prr_0.5={prr05_raw:.4f}, norm_0.5={prr05_norm:.4f}")
        else:
            print(f"{estimator}: No valid data or data length mismatch")

    
    if alignscores:
        oracle_ue = 1.0 - np.array(alignscores)
        rejection_ratios, oracle_curve = compute_curve(oracle_ue, alignscores)

        prr_results['Oracle'] = {
            'rejection_ratios': rejection_ratios.tolist(),
            'alignscore_means': oracle_curve,
            'prr': calculate_prr_area(oracle_ue, alignscores, 1.0),
            'prr_norm': 1.0,  
            'prr_0.5': calculate_prr_area(oracle_ue, alignscores, 0.5),
            'prr_norm_0.5': 1.0,
        }

    
    dir = 'max_match_giga_7b_11_11'
    os.makedirs(dir, exist_ok=True)
    
    if prr_results:
        plot_metric_with_oracle(prr_results, 'prr', dir+'/prr_curves.png', 'PRR Raw', top_n=8)
        plot_metric_with_oracle(prr_results, 'prr_norm', dir+'/prr_norm_curves.png', 'PRR Normalized', top_n=8)
        plot_metric_with_oracle(prr_results, 'prr_0.5', dir+'/prr05_curves.png', 'PRR (0.5)', top_n=8)
        plot_metric_with_oracle(prr_results, 'prr_norm_0.5', dir+'/prr_norm05_curves.png', 'PRR Normalized (0.5)', top_n=8)

    with open(dir+"/prr_results_max_matching.json", 'w') as f:
        json.dump(prr_results, f, indent=4)

    plot_prr_ranking(prr_results, "prr", dir+"/ranking_prr.png", "PRR Raw")
    plot_prr_ranking(prr_results, "prr_norm", dir+"/ranking_prr_norm.png", "PRR Normalized")
    plot_prr_ranking(prr_results, 'prr_0.5', dir+'/ranking_prr_05.png', 'PRR (0.5)')
    plot_prr_ranking(prr_results, 'prr_norm_0.5', dir+'/ranking_prr_norm_05.png', 'PRR Normalized (0.5)')

    print("\nPRR calculation + plots completed!")

if __name__ == "__main__":
    main()

In [ ]:
#norma 4 graphics ORACLE
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
import os

def extract_alignscores(results: List[Dict]) -> List[Dict]:
    valid_results = []
    for res in results:
        if 'estimations' in res and 'AlignScoreMulti' in res['estimations']:
            
            res['min_matching_score'] = res['estimations']['AlignScoreMulti']
            valid_results.append(res)
    return valid_results

# --------------------------
# Normalization helpers
# --------------------------
def normalize(target: List[float]):
    min_t, max_t = np.min(target), np.max(target)
    if np.isclose(min_t, max_t):
        min_t -= 1
        max_t += 1
    target = (np.array(target) - min_t) / (max_t - min_t)
    return target

def get_random_scores(function, metrics, num_iter=200, seed=42):
    np.random.seed(seed)
    rand_scores = np.arange(len(metrics))
    value = []
    for _ in range(num_iter):
        np.random.shuffle(rand_scores)
        rand_val = function(rand_scores, metrics)
        value.append(rand_val)
    return np.mean(value)

def normalize_metric(target_score, oracle_score, random_score):
    if not (oracle_score == random_score):
        target_score = (target_score - random_score) / (oracle_score - random_score)
    return target_score

# --------------------------
# Core PRR calc
# --------------------------
def calculate_prr_area(estimator: List[float], target: List[float], max_rejection: float = 1.0) -> float:
    target = normalize(target)
    ue = np.array(estimator)
    num_obs = len(ue)
    num_rej = int(max_rejection * num_obs)

    ue_argsort = np.argsort(ue)
    sorted_metrics = np.array(target)[ue_argsort]
    cumsum = np.cumsum(sorted_metrics)[-num_rej:]
    scores = (cumsum / np.arange((num_obs - num_rej) + 1, num_obs + 1))[::-1]
    prr_score = np.sum(scores) / num_rej
    return prr_score

def compute_prr_and_normalized(estimator: List[float], target: List[float], max_rejection: float = 1.0):
    raw_score = calculate_prr_area(estimator, target, max_rejection)
    oracle_ue = 1.0 - normalize(target)
    oracle_score = calculate_prr_area(oracle_ue, target, max_rejection)
    random_score = get_random_scores(
        lambda e, t: calculate_prr_area(e, t, max_rejection), target
    )
    norm_score = normalize_metric(raw_score, oracle_score, random_score)
    return raw_score, norm_score

def compute_curve(estimator, target, max_rejection=1.0, num_points=95, do_normalize=False):
    target = normalize(target) if do_normalize else np.asarray(target)
    ue = np.asarray(estimator)
    num_obs = len(ue)

    
    sort_idx = np.argsort(ue)
    sorted_metrics = target[sort_idx]

    rejection_ratios = np.linspace(0, max_rejection, num_points)
    alignscore_means = []
    for rr in rejection_ratios:
        n_keep = int(num_obs * (1 - rr))
        alignscore_means.append(np.mean(sorted_metrics[:n_keep]) if n_keep > 0 else 0.0)
    return rejection_ratios, alignscore_means

# --------------------------
# Plotting functions
# --------------------------
def plot_metric_with_oracle(prr_results, key, output_file, ylabel, top_n=8):
    


    plt.figure(figsize=(12, 8))
    
    
    sorted_items = sorted(
        [(name, vals) for name, vals in prr_results.items() if name != "Oracle" and key in vals],
        key=lambda x: x[1][key],
        reverse=True
    )
    if not sorted_items:
        print(f"[warn] nothing to plot for key={key}")
        return
    
    top_estimators = [name for name, _ in sorted_items[:top_n]]
    colors = plt.cm.Set3(np.linspace(0, 1, min(top_n, len(top_estimators))))
    
    
    for i, name in enumerate(top_estimators):
        vals = prr_results[name]
        plt.plot(vals['rejection_ratios'], vals['alignscore_means'],
                 '-', linewidth=3, color=colors[i],
                 label=f"{name} ({key}={vals[key]:.3f})", alpha=0.85)
    
    
    anchor = top_estimators[0]
    anchor_vals = prr_results[anchor]
    
    oracle_ue = 1.0 - np.asarray(anchor_vals['alignscores_raw'])
    rr_oracle, curve_oracle = compute_curve(
        oracle_ue,
        anchor_vals['alignscores_raw'],
        max_rejection=1.0,
        num_points=len(anchor_vals['rejection_ratios']),
        do_normalize=False
    )
    
    plt.plot(rr_oracle, curve_oracle,
             '-', linewidth=3.5, color='black',
             label=f"Oracle ({key}={prr_results['Oracle'][key]:.3f})")
    
    plt.title(f'PRR Curves - {ylabel}', fontsize=14, fontweight='bold')
    plt.xlabel('Rejection Ratio', fontsize=12)
    plt.ylabel('AlignScoreMulti', fontsize=12)  
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 0.9)
    plt.ylim(0.5, 1.0)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()

def plot_prr_ranking(prr_results: Dict[str, Dict], key: str, output_file: str, ylabel: str, top_n: int = 20):
    sorted_estimators = sorted(prr_results.items(), key=lambda x: x[1][key], reverse=True)
    top_estimators = sorted_estimators[:top_n]

    names = [name for name, _ in top_estimators]
    values = [data[key] for _, data in top_estimators]

    plt.figure(figsize=(10, 6))
    bars = plt.barh(range(len(names)), values, color=plt.cm.Set3(np.linspace(0, 1, len(names))))
    plt.yticks(range(len(names)), names, fontsize=9)
    plt.xlabel(ylabel, fontsize=12)
    plt.title(f"Top-{top_n} estimators by {ylabel}", fontsize=14, fontweight="bold")
    plt.gca().invert_yaxis()  

    for i, bar in enumerate(bars):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{values[i]:.3f}", va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

# --------------------------
# Main
# --------------------------
def main():
    log_file = '../rvanova/giga1/Lite_v2/hf/rvanova/med/2025-10-18/12-36-44/polygraph_eval.log'
    with open(log_file, 'r', encoding='utf-8') as f:
        log_content = f.read()

    results = parse_log_file(log_content)
    results_with_alignscores = extract_alignscores(results)
    
    if not results_with_alignscores:
        print("No valid results found with min_matching_score.")
        return

    print(f"Found {len(results_with_alignscores)} results with min_matching_score")
    
    
    all_estimators = set()
    for res in results_with_alignscores:
        if 'estimations' in res:
            all_estimators.update(res['estimations'].keys())
    
    
    relevant_estimators = [
        'MaximumSequenceProbability', 'Perplexity', 'MeanTokenEntropy',
        'MeanPointwiseMutualInformation', 'SelfCertainty',
        'MeanConditionalPointwiseMutualInformation', 'CCP', 'PTrue',
        'PTrueSampling', 'MonteCarloSequenceEntropy',
        'MonteCarloNormalizedSequenceEntropy', 'LexicalSimilarity_rouge1',
        'LexicalSimilarity_rouge2', 'LexicalSimilarity_rougeL',
        'LexicalSimilarity_BLEU', 'NumSemSets', 'EigValLaplacian_NLI_score_entail',
        'DegMat_NLI_score_entail', 'DegMat_NLI_score_contra', 'DegMat_Jaccard_score',
        'SemanticEntropy', 'LUQ', 'KernelLanguageEntropy', 'EigenScore',
        'MahalanobisDistanceSeq_decoder', 'RelativeMahalanobisDistanceSeq_decoder',
        'RDESeq_decoder', 'PPLMDSeq_decoder', 'PPLRMDSeq_decoder',
        'AttentionScore (layer=14)', 'RAUQ', 'RAUQ (entropy)', 'CSL',
        'CocoaMSP', 'CocoaPPL', 'CocoaMTE', 'SemanticDensity'
    ]
    print(f"Available estimators: {relevant_estimators}")
    
    prr_results = {}
    alignscores = []  
    
    for estimator in relevant_estimators:
        uncertainties, matching_scores = [], []
        for res in results_with_alignscores:
            if estimator in res['estimations'] and 'min_matching_score' in res:
                uncertainties.append(res['estimations'][estimator])
                matching_scores.append(res['min_matching_score'])

        if uncertainties and len(uncertainties) == len(matching_scores):
            alignscores = matching_scores  
            prr_raw, prr_norm = compute_prr_and_normalized(uncertainties, matching_scores, 1.0)
            prr05_raw, prr05_norm = compute_prr_and_normalized(uncertainties, matching_scores, 0.5)

            sort_idx = np.argsort(uncertainties)
            sorted_matching_scores = np.array(matching_scores)[sort_idx]
            rejection_ratios = np.linspace(0, 1.0, 95)
            alignscore_means = [
                np.mean(sorted_matching_scores[:int(len(sorted_matching_scores)*(1-r))])
                if int(len(sorted_matching_scores)*(1-r)) > 0 else 0
                for r in rejection_ratios
            ]

            prr_results[estimator] = {
                'rejection_ratios': rejection_ratios.tolist(),
                'alignscore_means': alignscore_means,
                'prr': prr_raw,
                'prr_norm': prr_norm,
                'prr_0.5': prr05_raw,
                'prr_norm_0.5': prr05_norm,
                'alignscores_raw': matching_scores,      
                'uncertainties_raw': uncertainties       
            }
            print(f"{estimator}: prr={prr_raw:.4f}, norm={prr_norm:.4f}, "
                  f"prr_0.5={prr05_raw:.4f}, norm_0.5={prr05_norm:.4f}")
        else:
            print(f"{estimator}: No valid data or data length mismatch")

    
    if alignscores:
        oracle_ue = 1.0 - np.array(alignscores)
        rejection_ratios, oracle_curve = compute_curve(oracle_ue, alignscores)

        prr_results['Oracle'] = {
            'rejection_ratios': rejection_ratios.tolist(),
            'alignscore_means': oracle_curve,
            'prr': calculate_prr_area(oracle_ue, alignscores, 1.0),
            'prr_norm': 1.0,  
            'prr_0.5': calculate_prr_area(oracle_ue, alignscores, 0.5),
            'prr_norm_0.5': 1.0,
        }

    
    dir = 'align_giga_7b_11_11'
    os.makedirs(dir, exist_ok=True)
    
    if prr_results:
        plot_metric_with_oracle(prr_results, 'prr', dir+'/prr_curves.png', 'PRR Raw', top_n=8)
        plot_metric_with_oracle(prr_results, 'prr_norm', dir+'/prr_norm_curves.png', 'PRR Normalized', top_n=8)
        plot_metric_with_oracle(prr_results, 'prr_0.5', dir+'/prr05_curves.png', 'PRR (0.5)', top_n=8)
        plot_metric_with_oracle(prr_results, 'prr_norm_0.5', dir+'/prr_norm05_curves.png', 'PRR Normalized (0.5)', top_n=8)

    with open(dir+"/prr_results_min_matching.json", 'w') as f:
        json.dump(prr_results, f, indent=4)

    plot_prr_ranking(prr_results, "prr", dir+"/ranking_prr.png", "PRR Raw")
    plot_prr_ranking(prr_results, "prr_norm", dir+"/ranking_prr_norm.png", "PRR Normalized")
    plot_prr_ranking(prr_results, 'prr_0.5', dir+'/ranking_prr_05.png', 'PRR (0.5)')
    plot_prr_ranking(prr_results, 'prr_norm_0.5', dir+'/ranking_prr_norm_05.png', 'PRR Normalized (0.5)')

    print("\nPRR calculation + plots completed!")

if __name__ == "__main__":
    main()